# Ling 3.0 Flash VL INT4 — Zero-shot VQA

Qwen3.5-122B 검증 노트북과 같은 목적의 **깡모델 zero-shot** 실행본입니다.

- 모델: `inclusionAI/Ling-3.0-flash-VL-int4`
- 권장 런타임: **G4 96GB** (A100 80GB는 여유가 작아 비권장)
- 학습/LoRA 없음
- OCR/DINO 없음
- 외부 API 없음
- 실제 이미지 + 질문 + a/b/c/d
- thinking 비활성화
- structured output으로 `a/b/c/d`만 허용
- Test 6,714개 resume + submission 생성

> Ling 3.0 VL은 일반 vLLM보다 InclusionAI의 Ling 지원 vLLM fork를 사용합니다.


## Step 0. Ling vLLM 환경 설치

이 셀 실행 후 **런타임을 한 번 재시작**하고 Step 1부터 진행하세요.


In [2]:
# ============================================================
# Ling-3.0 Flash VL INT4
# STEP 0 FINAL SOURCE BUILD — G4 / CUDA 13.0
#
# 핵심:
#   ❌ VLLM_USE_PRECOMPILED 사용 안 함
#   ✅ Ling fork Python + C++/CUDA extension 동일 소스에서 build
#
# 예상 시간: 꽤 걸릴 수 있음
# 끝나면 반드시 세션 재시작
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path


def run(cmd, env=None, cwd=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)

    subprocess.run(
        [str(x) for x in cmd],
        check=True,
        env=env,
        cwd=cwd,
    )


print("=" * 80)
print("LING SOURCE BUILD STEP 0")
print("=" * 80)


# ============================================================
# 1. uv / build tools
# ============================================================

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "uv",
    "ninja",
    "cmake",
    "packaging",
])


# ============================================================
# 2. CUDA 13.0 toolkit compiler 설치
#
# nvidia 공식 PyPI CUDA toolkit meta-package
# nvcc 13.0.88 포함
# ============================================================

print("\n[1/7] CUDA 13.0 compiler/toolkit 설치")

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "cuda-toolkit[nvcc,crt,cccl]==13.0.3",
])


# ============================================================
# 3. CUDA_HOME / nvcc 위치 자동 탐색
# ============================================================

print("\n[2/7] CUDA compiler 탐색")

import site

candidates = []

# 일반 CUDA 설치
for p in [
    "/usr/local/cuda",
    "/usr/local/cuda-13.0",
]:
    if Path(p, "bin", "nvcc").exists():
        candidates.append(Path(p))


# pip NVIDIA CUDA 설치 위치 탐색
for base in site.getsitepackages():

    base = Path(base)

    for p in [
        base / "nvidia" / "cu13",
        base / "nvidia" / "cuda_nvcc",
        base / "nvidia" / "cuda_nvcc" / "bin",
    ]:

        if (p / "bin" / "nvcc").exists():
            candidates.append(p)

        elif (p / "nvcc").exists():
            # bin directory 자체
            candidates.append(p.parent)


# PATH에서도 탐색
which_nvcc = shutil.which("nvcc")

if which_nvcc:
    nvcc_path = Path(which_nvcc).resolve()
    candidates.append(nvcc_path.parent.parent)


CUDA_HOME = None

for p in candidates:

    nvcc = p / "bin" / "nvcc"

    if nvcc.exists():
        CUDA_HOME = p
        break


if CUDA_HOME is None:

    # pip package 위치 전체 탐색 fallback
    found = []

    for base in site.getsitepackages():

        for nvcc in Path(base).rglob("nvcc"):

            if (
                nvcc.is_file()
                and nvcc.parent.name == "bin"
            ):
                found.append(nvcc)

    print("nvcc candidates:", found[:20])

    if not found:
        raise RuntimeError(
            "CUDA nvcc를 찾지 못했습니다."
        )

    nvcc = found[0]

    CUDA_HOME = (
        nvcc.parent.parent
    )


NVCC = (
    CUDA_HOME
    / "bin"
    / "nvcc"
)


print("CUDA_HOME:", CUDA_HOME)
print("NVCC     :", NVCC)

assert NVCC.exists()


# 환경 변수
os.environ["CUDA_HOME"] = str(CUDA_HOME)

os.environ["PATH"] = (
    f"{CUDA_HOME}/bin:"
    + os.environ["PATH"]
)


# ============================================================
# 4. NVCC 실제 버전 출력
# ============================================================

print("\n[3/7] NVCC VERSION")

result = subprocess.run(
    [
        str(NVCC),
        "--version",
    ],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)
print(result.stderr)

combined = (
    result.stdout
    + result.stderr
)

if (
    "release 13." not in combined
    and "release 12.9" not in combined
):
    raise RuntimeError(
        "NVCC 12.9+가 필요합니다.\n"
        + combined
    )


# ============================================================
# 5. 기존 vLLM / Ling 제거
# ============================================================

print("\n[4/7] 기존 precompiled vLLM 제거")

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "uninstall",
    "-y",
    "vllm",
],
check=False)


REPO = Path(
    "/content/vllm-ling-v3"
)

if REPO.exists():
    shutil.rmtree(REPO)


# ============================================================
# 6. Ling fork fresh clone
# ============================================================

print("\n[5/7] Ling vLLM fork clone")

run([
    "git",
    "clone",
    "--depth",
    "1",
    "https://github.com/inclusionAI/vllm-ling-v3.git",
    str(REPO),
])


# ============================================================
# 7. build dependencies
#
# 현재 Torch 2.13 cu130을 그대로 사용
# ============================================================

print("\n[6/7] Build dependencies 설치")

# 현재 torch 사용하도록 설정
run([
    sys.executable,
    str(
        REPO
        / "tools"
        / "use_existing_torch.py"
    ),
],
cwd=REPO)


run([
    "uv",
    "pip",
    "install",
    "--system",
    "-r",
    str(
        REPO
        / "requirements"
        / "build"
        / "cuda.txt"
    ),
])


# ============================================================
# 8. FULL SOURCE BUILD
#
# 중요:
# VLLM_USE_PRECOMPILED 설정 없음
# ============================================================

print("\n[7/7] Ling vLLM FULL SOURCE BUILD")
print("이 단계가 가장 오래 걸립니다.")

env = os.environ.copy()

env.pop(
    "VLLM_USE_PRECOMPILED",
    None
)

env.pop(
    "VLLM_PRECOMPILED_WHEEL_COMMIT",
    None
)

env.pop(
    "VLLM_PRECOMPILED_WHEEL_LOCATION",
    None
)

env["CUDA_HOME"] = str(
    CUDA_HOME
)

env["PATH"] = (
    f"{CUDA_HOME}/bin:"
    + env["PATH"]
)

# Colab CPU/RAM 과부하 방지
env["MAX_JOBS"] = "4"

# G4 Blackwell
# vLLM이 현재 GPU를 자동 인식하도록 비움
env.pop(
    "TORCH_CUDA_ARCH_LIST",
    None
)


run([
    "uv",
    "pip",
    "install",
    "--system",
    "--no-build-isolation",
    "--editable",
    str(REPO),
],
env=env,
cwd=REPO)


# ============================================================
# 9. fresh process ABI 검증
# ============================================================

print("\n" + "=" * 80)
print("ABI VERIFY")
print("=" * 80)


verify_code = r'''
import torch
import inspect
import vllm
import vllm._custom_ops as ops

print("vLLM:", vllm.__version__)
print("vLLM path:", ops.__file__)

print("\nPython wrapper:")
print(inspect.signature(ops.gptq_marlin_moe_repack))

print("\nCompiled C++ schema:")
print(torch.ops._C.gptq_marlin_repack._schemas)

schema = str(
    torch.ops._C.gptq_marlin_repack._schemas
)

# Ling fork는 perm을 사용하는 API를 요구
if "Tensor perm" not in schema and schema.count(",") < 5:
    raise RuntimeError(
        "아직 Python/C++ ABI mismatch입니다:\n"
        + schema
    )

print("\n✅ Python / C++ vLLM ABI가 source build 기준으로 로드됨")
'''


run([
    sys.executable,
    "-c",
    verify_code,
],
env=env)


print("\n" + "=" * 80)
print("🎉 SOURCE BUILD COMPLETE")
print("=" * 80)

print("""
이제 반드시:

    런타임 → 세션 다시 시작

그 다음 Ling STEP 1 모델 로드를 실행하세요.

중요:
이제부터 VLLM_USE_PRECOMPILED=1 설치 셀은
절대로 다시 실행하지 마세요.
""")

LING SOURCE BUILD STEP 0

$ /usr/bin/python3 -m pip install -U uv ninja cmake packaging

[1/7] CUDA 13.0 compiler/toolkit 설치

$ /usr/bin/python3 -m pip install -U cuda-toolkit[nvcc,crt,cccl]==13.0.3

[2/7] CUDA compiler 탐색
CUDA_HOME: /usr/local/cuda
NVCC     : /usr/local/cuda/bin/nvcc

[3/7] NVCC VERSION
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0




RuntimeError: NVCC 12.9+가 필요합니다.
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
# ============================================================
# LING SOURCE BUILD — CUDA 13 FIXED VERSION
#
# 현재 세션 그대로 실행
# /usr/local/cuda 12.8은 무시하고
# pip로 설치된 CUDA 13.x를 강제로 사용
# ============================================================

import os
import sys
import site
import shutil
import subprocess
from pathlib import Path


def run(cmd, cwd=None, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)

    subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        check=True,
    )


print("=" * 80)
print("LING CUDA13 SOURCE BUILD")
print("=" * 80)


# ============================================================
# 1. pip CUDA 13 NVCC 찾기
#
# 중요:
# /usr/local/cuda는 절대 우선하지 않음.
# ============================================================

nvcc_candidates = []

for sp in site.getsitepackages():

    sp = Path(sp)

    candidates = [
        # CUDA 13 unified wheel layout
        sp / "nvidia" / "cu13" / "bin" / "nvcc",

        # alternate NVIDIA nvcc wheel layout
        sp / "nvidia" / "cuda_nvcc" / "bin" / "nvcc",
    ]

    for p in candidates:

        if p.is_file():
            nvcc_candidates.append(p)


# fallback: site-packages 전체에서 CUDA nvcc만 검색
if not nvcc_candidates:

    for sp in site.getsitepackages():

        sp = Path(sp)

        for p in sp.glob(
            "nvidia/**/bin/nvcc"
        ):

            if p.is_file():
                nvcc_candidates.append(p)


if not nvcc_candidates:
    raise RuntimeError(
        "pip CUDA 13 nvcc를 찾지 못했습니다."
    )


print("\nNVCC candidates:")

for p in nvcc_candidates:
    print(" -", p)


# ============================================================
# 2. 실제 버전 확인 후 13.x만 선택
# ============================================================

NVCC = None
NVCC_OUTPUT = None

for candidate in nvcc_candidates:

    result = subprocess.run(
        [str(candidate), "--version"],
        capture_output=True,
        text=True,
    )

    output = (
        result.stdout
        + "\n"
        + result.stderr
    )

    print(
        "\nCHECK:",
        candidate
    )

    print(output)

    if "release 13." in output:

        NVCC = candidate.resolve()
        NVCC_OUTPUT = output

        break


if NVCC is None:

    raise RuntimeError(
        "CUDA 13.x nvcc를 찾았지만 "
        "실제 버전 검증에 실패했습니다."
    )


print("\n✅ CUDA 13 NVCC FOUND")
print("NVCC:", NVCC)


# ============================================================
# 3. CUDA_HOME 결정
#
# 예상:
# .../site-packages/nvidia/cu13/bin/nvcc
#
# CUDA_HOME:
# .../site-packages/nvidia/cu13
# ============================================================

CUDA_HOME = NVCC.parent.parent

print(
    "CUDA_HOME:",
    CUDA_HOME
)


# 필수 디렉터리 확인
print("\nCUDA_HOME contents:")

for name in [
    "bin",
    "include",
    "lib",
    "lib64",
]:

    p = CUDA_HOME / name

    print(
        f"{name:8s}:",
        p,
        p.exists()
    )


# ============================================================
# 4. 환경변수 CUDA 13으로 강제
# ============================================================

os.environ["CUDA_HOME"] = str(
    CUDA_HOME
)

os.environ["CUDACXX"] = str(
    NVCC
)

os.environ["PATH"] = (
    f"{NVCC.parent}:"
    + os.environ["PATH"]
)


# CUDA13 pip header paths
include_paths = []

for p in [
    CUDA_HOME / "include",
    CUDA_HOME / "include" / "cccl",
]:

    if p.exists():
        include_paths.append(
            str(p)
        )


if include_paths:

    old = os.environ.get(
        "CPATH",
        ""
    )

    os.environ["CPATH"] = (
        ":".join(include_paths)
        + (":" + old if old else "")
    )


# CUDA13 pip lib paths
lib_paths = []

for p in [
    CUDA_HOME / "lib",
    CUDA_HOME / "lib64",
]:

    if p.exists():
        lib_paths.append(
            str(p)
        )


if lib_paths:

    old = os.environ.get(
        "LIBRARY_PATH",
        ""
    )

    os.environ["LIBRARY_PATH"] = (
        ":".join(lib_paths)
        + (":" + old if old else "")
    )

    old = os.environ.get(
        "LD_LIBRARY_PATH",
        ""
    )

    os.environ["LD_LIBRARY_PATH"] = (
        ":".join(lib_paths)
        + (":" + old if old else "")
    )


# ============================================================
# 5. 진짜 nvcc가 바뀌었는지 확인
# ============================================================

print("\n" + "=" * 80)
print("CUDA FINAL CHECK")
print("=" * 80)

print(
    "which nvcc:",
    shutil.which("nvcc")
)

print(
    "CUDA_HOME :",
    os.environ["CUDA_HOME"]
)

print(
    "CUDACXX   :",
    os.environ["CUDACXX"]
)


result = subprocess.run(
    ["nvcc", "--version"],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

assert "release 13." in result.stdout, (
    "PATH가 아직 CUDA 12.8을 가리킵니다."
)


print("✅ NVCC 13.x ACTIVE")


# ============================================================
# 6. Torch도 cu130인지 확인
# ============================================================

import torch

print("\nTorch      :", torch.__version__)
print("Torch CUDA :", torch.version.cuda)

assert torch.version.cuda == "13.0"

print("✅ Torch cu130 / NVCC 13.x 조합 확인")


# ============================================================
# 7. build tools
# ============================================================

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "uv",
    "ninja",
    "cmake",
    "packaging",
])


# ============================================================
# 8. 기존 precompiled vLLM 제거
# ============================================================

print("\n" + "=" * 80)
print("REMOVE PRECOMPILED VLLM")
print("=" * 80)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "vllm",
    ],
    check=False,
)


# ============================================================
# 9. Ling fork fresh clone
# ============================================================

REPO = Path(
    "/content/vllm-ling-v3"
)

if REPO.exists():

    shutil.rmtree(
        REPO
    )


run([
    "git",
    "clone",
    "--depth",
    "1",
    "https://github.com/inclusionAI/vllm-ling-v3.git",
    str(REPO),
])


# ============================================================
# 10. 기존 Torch 그대로 사용하도록 requirements 수정
# ============================================================

print("\n" + "=" * 80)
print("USE EXISTING TORCH")
print("=" * 80)


# 버전에 따라 위치가 다를 수 있으므로 자동 탐색
torch_script_candidates = [
    REPO / "use_existing_torch.py",
    REPO / "tools" / "use_existing_torch.py",
]

torch_script = None

for p in torch_script_candidates:

    if p.exists():

        torch_script = p
        break


if torch_script is None:

    raise RuntimeError(
        "use_existing_torch.py를 찾지 못했습니다."
    )


print(
    "use_existing_torch:",
    torch_script
)


run([
    sys.executable,
    str(torch_script),
],
cwd=REPO)


# ============================================================
# 11. build requirements
# ============================================================

BUILD_REQ = (
    REPO
    / "requirements"
    / "build"
    / "cuda.txt"
)

assert BUILD_REQ.exists(), (
    f"build requirements 없음: {BUILD_REQ}"
)


run([
    "uv",
    "pip",
    "install",
    "--system",
    "-r",
    str(BUILD_REQ),
])


# ============================================================
# 12. Full source build
#
# 절대 precompiled 사용하지 않음
# ============================================================

print("\n" + "=" * 80)
print("FULL VLLM-LING SOURCE BUILD")
print("=" * 80)

print(
    "CUDA_HOME:",
    CUDA_HOME
)

print(
    "NVCC:",
    NVCC
)

print(
    "이제 C++ / CUDA extension까지 직접 빌드합니다."
)


env = os.environ.copy()

# 이전 precompiled 설정 완전 제거
for key in [
    "VLLM_USE_PRECOMPILED",
    "VLLM_PRECOMPILED_WHEEL_COMMIT",
    "VLLM_PRECOMPILED_WHEEL_LOCATION",
]:

    env.pop(
        key,
        None
    )


env["CUDA_HOME"] = str(
    CUDA_HOME
)

env["CUDACXX"] = str(
    NVCC
)

env["PATH"] = (
    f"{NVCC.parent}:"
    + env["PATH"]
)

# G4 Colab CPU/RAM 폭주 방지
env["MAX_JOBS"] = "4"


run([
    "uv",
    "pip",
    "install",
    "--system",
    "--no-build-isolation",
    "--editable",
    ".",
],
cwd=REPO,
env=env)


# ============================================================
# 13. Fresh process ABI 검증
# ============================================================

print("\n" + "=" * 80)
print("ABI CHECK")
print("=" * 80)


verify_code = r'''
import torch
import inspect
import vllm
import vllm._custom_ops as ops

print("vLLM:", vllm.__version__)
print("vLLM path:", ops.__file__)

print("\nPython wrapper:")
print(
    inspect.signature(
        ops.gptq_marlin_moe_repack
    )
)

print("\nCompiled op:")
schema = (
    torch.ops._C
    .gptq_marlin_repack
    ._schemas
)

print(schema)

wrapper = str(
    inspect.signature(
        ops.gptq_marlin_moe_repack
    )
)

schema_text = str(schema)

print("\n--- ABI RESULT ---")

# 이전 깨진 상태:
#
# Python:
#   b_q_weight, perm, size_k, ...
#
# C++:
#   b_q_weight, size_k, ...
#
# source build 후에는 Python/C++가 서로 대응해야 함.

if (
    "perm" in wrapper
    and "perm" not in schema_text
):
    raise RuntimeError(
        "❌ ABI mismatch가 아직 남아있습니다.\n"
        f"Python = {wrapper}\n"
        f"C++ = {schema_text}"
    )

print("✅ Python / C++ ABI MATCH")
'''


run([
    sys.executable,
    "-c",
    verify_code,
],
env=env)


print("\n" + "=" * 80)
print("🎉 LING SOURCE BUILD SUCCESS")
print("=" * 80)

print("""
다음 단계:

1. 런타임 → 세션 다시 시작
2. STEP 0 다시 실행하지 않기
3. Ling STEP 1 모델 로드 실행

중요:
VLLM_USE_PRECOMPILED=1 셀은 앞으로 실행하지 마세요.
""")

LING CUDA13 SOURCE BUILD

NVCC candidates:
 - /usr/local/lib/python3.13/dist-packages/nvidia/cu13/bin/nvcc

CHECK: /usr/local/lib/python3.13/dist-packages/nvidia/cu13/bin/nvcc
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0



✅ CUDA 13 NVCC FOUND
NVCC: /usr/local/lib/python3.13/dist-packages/nvidia/cu13/bin/nvcc
CUDA_HOME: /usr/local/lib/python3.13/dist-packages/nvidia/cu13

CUDA_HOME contents:
bin     : /usr/local/lib/python3.13/dist-packages/nvidia/cu13/bin True
include : /usr/local/lib/python3.13/dist-packages/nvidia/cu13/include True
lib     : /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib True
lib64   : /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib64 False

CUDA FINAL CHECK
which nvcc: /usr/local/lib/python3.13/dist-packages/nvidia/cu13/bin/nvcc
CUDA_HOME : /usr/local/lib/python3.13/dist-packages/nvid

CalledProcessError: Command '['uv', 'pip', 'install', '--system', '--no-build-isolation', '--editable', '.']' returned non-zero exit status 1.

In [4]:
# ============================================================
# LING SOURCE BUILD — VERBOSE DIAGNOSTIC + RETRY
#
# 현재 세션 그대로 실행
# 재시작 X
# clone X
# requirements 재설치 X
#
# 목적:
#   1. CUDA 13 환경 다시 고정
#   2. GCC / CMake / nvcc 확인
#   3. vLLM-Ling source build를 VERBOSE로 실행
#   4. 실패하면 실제 compiler error 마지막 250줄 출력
#   5. 성공하면 Marlin ABI 자동 검증
# ============================================================

import os
import sys
import subprocess
import shutil
from pathlib import Path


REPO = Path("/content/vllm-ling-v3")

CUDA_HOME = Path(
    "/usr/local/lib/python3.13/dist-packages/nvidia/cu13"
)

NVCC = CUDA_HOME / "bin" / "nvcc"

LOG_PATH = Path(
    "/content/ling_vllm_source_build_verbose.log"
)


print("=" * 80)
print("LING SOURCE BUILD VERBOSE RETRY")
print("=" * 80)


# ============================================================
# 1. 기본 경로 확인
# ============================================================

assert REPO.exists(), (
    f"Repo 없음: {REPO}"
)

assert NVCC.exists(), (
    f"CUDA 13 nvcc 없음: {NVCC}"
)

print("REPO     :", REPO)
print("CUDA_HOME:", CUDA_HOME)
print("NVCC     :", NVCC)


# ============================================================
# 2. 환경 다시 강제
# ============================================================

env = os.environ.copy()

env["CUDA_HOME"] = str(CUDA_HOME)
env["CUDACXX"] = str(NVCC)

env["PATH"] = (
    f"{CUDA_HOME}/bin:"
    + env.get("PATH", "")
)

# CMake도 CUDA 13 root를 명확히 알게 함
env["CUDAToolkit_ROOT"] = str(CUDA_HOME)

# pip CUDA toolkit include/lib
include_dir = CUDA_HOME / "include"
lib_dir = CUDA_HOME / "lib"

if include_dir.exists():

    old = env.get("CPATH", "")

    env["CPATH"] = (
        str(include_dir)
        + (":" + old if old else "")
    )

if lib_dir.exists():

    for key in [
        "LIBRARY_PATH",
        "LD_LIBRARY_PATH",
    ]:

        old = env.get(key, "")

        env[key] = (
            str(lib_dir)
            + (":" + old if old else "")
        )


# precompiled 바이너리 절대 금지
for key in [
    "VLLM_USE_PRECOMPILED",
    "VLLM_PRECOMPILED_WHEEL_COMMIT",
    "VLLM_PRECOMPILED_WHEEL_LOCATION",
    "VLLM_PRECOMPILED_WHEEL_VARIANT",
]:

    env.pop(key, None)


# 컴파일 병렬도 낮춤
# 원인 파악 + RAM 폭주 방지
env["MAX_JOBS"] = "2"

# CUDA build 명시
env["VLLM_TARGET_DEVICE"] = "cuda"


# ============================================================
# 3. lib64가 없으면 lib -> lib64 symlink
#
# 일부 CMake가 lib64를 기대하는 경우 대비
# ============================================================

lib64_dir = CUDA_HOME / "lib64"

if (
    not lib64_dir.exists()
    and lib_dir.exists()
):

    try:

        lib64_dir.symlink_to(
            lib_dir,
            target_is_directory=True
        )

        print(
            "✅ lib64 -> lib symlink 생성:",
            lib64_dir
        )

    except Exception as e:

        print(
            "lib64 symlink 생략:",
            repr(e)
        )


# ============================================================
# 4. Toolchain 정보
# ============================================================

def show(cmd):

    print(
        "\n$",
        " ".join(map(str, cmd))
    )

    r = subprocess.run(
        [str(x) for x in cmd],
        env=env,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(r.stdout)

    return r.returncode


print("\n" + "=" * 80)
print("TOOLCHAIN CHECK")
print("=" * 80)

show([str(NVCC), "--version"])
show(["gcc", "--version"])
show(["g++", "--version"])
show(["cmake", "--version"])
show(["ninja", "--version"])

print(
    "which nvcc:",
    shutil.which(
        "nvcc",
        path=env["PATH"]
    )
)


# ============================================================
# 5. Torch 확인
# ============================================================

check_torch = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import torch; "
            "print('torch=', torch.__version__); "
            "print('torch cuda=', torch.version.cuda)"
        ),
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(
    "\n",
    check_torch.stdout
)

if "13.0" not in check_torch.stdout:

    raise RuntimeError(
        "현재 Python의 torch가 cu130이 아닙니다."
    )


# ============================================================
# 6. 실제 source build
#
# -v를 켜고 stdout+stderr 전부 저장
# ============================================================

print("\n" + "=" * 80)
print("FULL SOURCE BUILD START")
print("=" * 80)

print(
    "로그 저장:",
    LOG_PATH
)

cmd = [
    "uv",
    "-v",
    "pip",
    "install",
    "--system",
    "--no-build-isolation",
    "--editable",
    ".",
]


with LOG_PATH.open(
    "w",
    encoding="utf-8",
    errors="replace",
) as logfile:

    process = subprocess.Popen(
        cmd,
        cwd=REPO,
        env=env,

        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,

        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    lines = []

    for line in process.stdout:

        print(
            line,
            end=""
        )

        logfile.write(line)
        logfile.flush()

        lines.append(line)

        # RAM에 로그 무한 저장 방지
        if len(lines) > 1000:
            lines = lines[-1000:]

    return_code = process.wait()


print("\n" + "=" * 80)
print(
    "BUILD RETURN CODE:",
    return_code
)
print("=" * 80)


# ============================================================
# 7. 실패 시 진짜 에러 출력
# ============================================================

if return_code != 0:

    print("\n❌ SOURCE BUILD FAILED")
    print(
        "\n아래가 실제 build log 마지막 부분입니다."
    )

    all_lines = LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    tail = all_lines[-250:]

    print("\n" + "=" * 80)
    print("LAST 250 BUILD LOG LINES")
    print("=" * 80)

    for line in tail:
        print(line)

    print("\n" + "=" * 80)
    print(
        "LOG FILE:",
        LOG_PATH
    )
    print("=" * 80)

    raise RuntimeError(
        "Source build 실패. "
        "위 LAST 250 BUILD LOG LINES가 실제 원인입니다."
    )


# ============================================================
# 8. 성공했다면 fresh process ABI 검증
# ============================================================

print("\n✅ SOURCE BUILD COMMAND SUCCESS")

verify_code = r'''
import torch
import inspect
import vllm
import vllm._custom_ops as ops

print("=" * 72)
print("VLLM ABI VERIFY")
print("=" * 72)

print("vLLM:", vllm.__version__)
print("path:", ops.__file__)

wrapper = inspect.signature(
    ops.gptq_marlin_moe_repack
)

schema = (
    torch.ops._C
    .gptq_marlin_repack
    ._schemas
)

print("\nPython wrapper:")
print(wrapper)

print("\nCompiled C++ schema:")
print(schema)

wrapper_text = str(wrapper)
schema_text = str(schema)

if (
    "perm" in wrapper_text
    and "perm" not in schema_text
):
    raise RuntimeError(
        "\n❌ ABI mismatch가 아직 남아있음\n"
        f"Python: {wrapper}\n"
        f"C++: {schema}"
    )

print("\n✅ PYTHON / C++ ABI MATCH")
'''


verify = subprocess.run(
    [
        sys.executable,
        "-c",
        verify_code,
    ],
    env=env,
    cwd=REPO,

    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(
    verify.stdout
)

if verify.returncode != 0:

    raise RuntimeError(
        "빌드는 성공했지만 ABI 검증 실패"
    )


print("\n" + "=" * 80)
print("🎉 LING SOURCE BUILD + ABI CHECK SUCCESS")
print("=" * 80)

print("""
이게 뜨면:

런타임 → 세션 다시 시작

그 다음 모델 로드 STEP 1로 갑니다.
""")

LING SOURCE BUILD VERBOSE RETRY
REPO     : /content/vllm-ling-v3
CUDA_HOME: /usr/local/lib/python3.13/dist-packages/nvidia/cu13
NVCC     : /usr/local/lib/python3.13/dist-packages/nvidia/cu13/bin/nvcc
✅ lib64 -> lib symlink 생성: /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib64

TOOLCHAIN CHECK

$ /usr/local/lib/python3.13/dist-packages/nvidia/cu13/bin/nvcc --version
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


$ gcc --version
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



$ g++ --version
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source

RuntimeError: Source build 실패. 위 LAST 250 BUILD LOG LINES가 실제 원인입니다.

In [5]:
# ============================================================
# LING SOURCE BUILD FIX
# CUDA 13 pip-layout -> CMake CUDA_CUDART_LIBRARY 해결
#
# 현재 세션 그대로 실행
# 재시작 X
# clone X
# torch 재설치 X
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path


REPO = Path("/content/vllm-ling-v3")

CUDA_HOME = Path(
    "/usr/local/lib/python3.13/dist-packages/nvidia/cu13"
)

NVCC = CUDA_HOME / "bin" / "nvcc"

SITE_PACKAGES = Path(
    "/usr/local/lib/python3.13/dist-packages"
)

LOG_PATH = Path(
    "/content/ling_vllm_source_build_cudart_fix.log"
)


print("=" * 80)
print("LING CUDA CUDART FIX + SOURCE BUILD")
print("=" * 80)

assert REPO.exists(), REPO
assert NVCC.exists(), NVCC


# ============================================================
# 1. libcudart 위치 찾기
# ============================================================

print("\n[1] libcudart 탐색")

search_roots = [
    CUDA_HOME,
    SITE_PACKAGES / "nvidia" / "cuda_runtime",
    SITE_PACKAGES / "nvidia",
]

cudart_candidates = []

for root in search_roots:

    if not root.exists():
        continue

    for p in root.rglob("libcudart.so*"):

        if (
            p.is_file()
            or p.is_symlink()
        ):
            cudart_candidates.append(
                p.resolve()
            )


# 중복 제거
cudart_candidates = list(
    dict.fromkeys(
        cudart_candidates
    )
)


print(
    "발견된 libcudart:",
    len(cudart_candidates)
)

for p in cudart_candidates:
    print(" -", p)


if not cudart_candidates:

    raise RuntimeError(
        "libcudart.so를 찾지 못했습니다."
    )


# CUDA 13 경로 우선
preferred = [
    p
    for p in cudart_candidates
    if (
        "cu13" in str(p)
        or "cuda_runtime" in str(p)
    )
]


if preferred:
    REAL_CUDART = preferred[0]
else:
    REAL_CUDART = cudart_candidates[0]


print("\n선택한 libcudart:")
print(REAL_CUDART)


# ============================================================
# 2. CUDA_HOME/lib 및 lib64 정리
# ============================================================

CUDA_LIB = (
    CUDA_HOME
    / "lib"
)

CUDA_LIB.mkdir(
    parents=True,
    exist_ok=True
)


CUDA_LIB64 = (
    CUDA_HOME
    / "lib64"
)


# 이전에 잘못 생성됐을 수도 있으므로 상태 확인
if CUDA_LIB64.exists():

    print(
        "lib64 존재:",
        CUDA_LIB64
    )

else:

    CUDA_LIB64.symlink_to(
        CUDA_LIB,
        target_is_directory=True
    )

    print(
        "✅ lib64 -> lib 생성"
    )


# ============================================================
# 3. unversioned libcudart.so 보장
#
# FindCUDA가 libcudart.so.13만 보고
# libcudart.so를 못 잡는 상황 방지
# ============================================================

CUDART_LINK = (
    CUDA_LIB
    / "libcudart.so"
)


if CUDART_LINK.exists() or CUDART_LINK.is_symlink():

    print(
        "libcudart.so 이미 존재:",
        CUDART_LINK,
        "->",
        CUDART_LINK.resolve()
    )

else:

    CUDART_LINK.symlink_to(
        REAL_CUDART
    )

    print(
        "✅ libcudart.so symlink 생성:"
    )

    print(
        CUDART_LINK,
        "->",
        REAL_CUDART
    )


assert CUDART_LINK.exists()


# ============================================================
# 4. targets/x86_64-linux/lib 레이아웃도 생성
#
# 일부 CMake CUDA 탐색기가 이 구조를 기대
# ============================================================

TARGET_ROOT = (
    CUDA_HOME
    / "targets"
    / "x86_64-linux"
)

TARGET_LIB = (
    TARGET_ROOT
    / "lib"
)

TARGET_INCLUDE = (
    TARGET_ROOT
    / "include"
)


TARGET_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


if not TARGET_LIB.exists():

    TARGET_LIB.symlink_to(
        CUDA_LIB,
        target_is_directory=True
    )

    print(
        "✅ targets/x86_64-linux/lib -> CUDA lib"
    )


if not TARGET_INCLUDE.exists():

    TARGET_INCLUDE.symlink_to(
        CUDA_HOME / "include",
        target_is_directory=True
    )

    print(
        "✅ targets/x86_64-linux/include -> CUDA include"
    )


# ============================================================
# 5. 환경변수
# ============================================================

env = os.environ.copy()

env["CUDA_HOME"] = str(
    CUDA_HOME
)

env["CUDA_PATH"] = str(
    CUDA_HOME
)

env["CUDAToolkit_ROOT"] = str(
    CUDA_HOME
)

env["CUDA_TOOLKIT_ROOT_DIR"] = str(
    CUDA_HOME
)

env["CUDACXX"] = str(
    NVCC
)

env["PATH"] = (
    f"{CUDA_HOME}/bin:"
    + env.get(
        "PATH",
        ""
    )
)


# runtime library 탐색
lib_paths = [
    str(CUDA_LIB),
    str(TARGET_LIB),
]

old_ld = env.get(
    "LD_LIBRARY_PATH",
    ""
)

env["LD_LIBRARY_PATH"] = (
    ":".join(lib_paths)
    + (
        ":" + old_ld
        if old_ld
        else ""
    )
)


old_library = env.get(
    "LIBRARY_PATH",
    ""
)

env["LIBRARY_PATH"] = (
    ":".join(lib_paths)
    + (
        ":" + old_library
        if old_library
        else ""
    )
)


# ============================================================
# 6. CMAKE에 경로 직접 전달
#
# vLLM setup.py는 CMAKE_ARGS를 읽어서
# cmake 호출 뒤에 그대로 추가함.
# ============================================================

cmake_args = [

    f"-DCUDA_TOOLKIT_ROOT_DIR={CUDA_HOME}",

    f"-DCUDAToolkit_ROOT={CUDA_HOME}",

    f"-DCUDA_CUDART_LIBRARY={CUDART_LINK}",

    f"-DCMAKE_CUDA_COMPILER={NVCC}",

    (
        "-DCMAKE_LIBRARY_PATH="
        + ";".join(lib_paths)
    ),
]


env["CMAKE_ARGS"] = (
    " ".join(
        cmake_args
    )
)


# precompiled 절대 사용 안 함
for key in [
    "VLLM_USE_PRECOMPILED",
    "VLLM_PRECOMPILED_WHEEL_COMMIT",
    "VLLM_PRECOMPILED_WHEEL_LOCATION",
    "VLLM_PRECOMPILED_WHEEL_VARIANT",
]:

    env.pop(
        key,
        None
    )


env["MAX_JOBS"] = "2"

env["VLLM_TARGET_DEVICE"] = "cuda"


print("\n" + "=" * 80)
print("CUDA CONFIG")
print("=" * 80)

print(
    "CUDA_HOME:",
    CUDA_HOME
)

print(
    "NVCC:",
    NVCC
)

print(
    "CUDART:",
    CUDART_LINK.resolve()
)

print(
    "LD_LIBRARY_PATH:",
    env["LD_LIBRARY_PATH"]
)

print(
    "CMAKE_ARGS:"
)

print(
    env["CMAKE_ARGS"]
)


# ============================================================
# 7. libcudart 실제 확인
# ============================================================

print("\n" + "=" * 80)
print("CUDA FILE CHECK")
print("=" * 80)

subprocess.run(
    [
        str(NVCC),
        "--version",
    ],
    env=env,
    check=True,
)


print("\nCUDA lib contents:")

for p in sorted(
    CUDA_LIB.glob(
        "libcudart*"
    )
):

    print(
        " -",
        p,
        "->",
        p.resolve()
    )


# ============================================================
# 8. 이전 CMake 임시 build 흔적 삭제
#
# source 자체는 유지
# ============================================================

print("\n[2] 이전 build cache 제거")

for path in [

    REPO / "build",

    REPO / "_skbuild",

]:

    if path.exists():

        shutil.rmtree(
            path
        )

        print(
            "삭제:",
            path
        )


# ============================================================
# 9. SOURCE BUILD 재시도
# ============================================================

print("\n" + "=" * 80)
print("SOURCE BUILD RETRY")
print("=" * 80)

cmd = [
    "uv",
    "pip",
    "install",
    "--system",
    "--no-build-isolation",
    "--editable",
    ".",
]


with LOG_PATH.open(
    "w",
    encoding="utf-8",
    errors="replace",
) as logfile:

    process = subprocess.Popen(

        cmd,

        cwd=REPO,

        env=env,

        stdout=subprocess.PIPE,

        stderr=subprocess.STDOUT,

        text=True,

        bufsize=1,
    )

    recent_lines = []

    for line in process.stdout:

        print(
            line,
            end=""
        )

        logfile.write(
            line
        )

        logfile.flush()

        recent_lines.append(
            line
        )

        if len(recent_lines) > 500:

            recent_lines = (
                recent_lines[-500:]
            )


    return_code = (
        process.wait()
    )


print("\n" + "=" * 80)
print(
    "BUILD RETURN CODE:",
    return_code
)
print("=" * 80)


# ============================================================
# 10. 실패 시 실제 원인 출력
# ============================================================

if return_code != 0:

    log_lines = (
        LOG_PATH
        .read_text(
            encoding="utf-8",
            errors="replace",
        )
        .splitlines()
    )

    print(
        "\n❌ BUILD FAILED"
    )

    print("\n" + "=" * 80)
    print("LAST 200 LOG LINES")
    print("=" * 80)

    for line in log_lines[-200:]:

        print(line)


    raise RuntimeError(
        "Ling source build 실패. "
        "위 마지막 로그 확인."
    )


# ============================================================
# 11. build 성공 -> ABI 검증
# ============================================================

print(
    "\n✅ SOURCE BUILD SUCCESS"
)


verify_code = r'''
import torch
import inspect
import vllm
import vllm._custom_ops as ops

print("=" * 72)
print("MARLIN ABI VERIFY")
print("=" * 72)

print(
    "vLLM:",
    vllm.__version__
)

print(
    "vLLM:",
    ops.__file__
)


wrapper = (
    inspect.signature(
        ops.gptq_marlin_moe_repack
    )
)

schema = (
    torch.ops._C
    .gptq_marlin_repack
    ._schemas
)


print("\nPython wrapper:")
print(wrapper)

print("\nCompiled C++ op:")
print(schema)


wrapper_text = str(
    wrapper
)

schema_text = str(
    schema
)


if (
    "perm" in wrapper_text
    and "perm" not in schema_text
):

    raise RuntimeError(
        "\n❌ 아직 ABI mismatch\n"
        f"Python: {wrapper}\n"
        f"C++: {schema}"
    )


print(
    "\n✅ MARLIN PYTHON/C++ ABI MATCH"
)
'''


verify = subprocess.run(

    [
        sys.executable,
        "-c",
        verify_code,
    ],

    cwd=REPO,

    env=env,

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,
)


print(
    verify.stdout
)


if verify.returncode != 0:

    raise RuntimeError(
        "Source build는 성공했지만 "
        "ABI 검증 실패"
    )


print("\n" + "=" * 80)
print("🎉 LING SOURCE BUILD COMPLETE")
print("=" * 80)

print("""
성공하면:

1. 런타임 → 세션 다시 시작
2. STEP 0 다시 실행 X
3. Ling 모델 로드 STEP 1 실행
""")

LING CUDA CUDART FIX + SOURCE BUILD

[1] libcudart 탐색
발견된 libcudart: 2
 - /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcudart.so.13
 - /usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12

선택한 libcudart:
/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcudart.so.13
lib64 존재: /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib64
✅ libcudart.so symlink 생성:
/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcudart.so -> /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcudart.so.13
✅ targets/x86_64-linux/lib -> CUDA lib
✅ targets/x86_64-linux/include -> CUDA include

CUDA CONFIG
CUDA_HOME: /usr/local/lib/python3.13/dist-packages/nvidia/cu13
NVCC: /usr/local/lib/python3.13/dist-packages/nvidia/cu13/bin/nvcc
CUDART: /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcudart.so.13
LD_LIBRARY_PATH: /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib:/usr/local/lib/python3.13/dist-packages/nvidia/cu13/targets/x8

RuntimeError: Ling source build 실패. 위 마지막 로그 확인.

In [6]:
# ============================================================
# LING SOURCE BUILD FIX v2
# NVRTC + CUDA DRIVER LIBRARY FIX
#
# 현재 세션 그대로 실행
# 재시작 X
# clone X
# torch 재설치 X
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path


REPO = Path("/content/vllm-ling-v3")

CUDA_HOME = Path(
    "/usr/local/lib/python3.13/dist-packages/nvidia/cu13"
)

CUDA_BIN = CUDA_HOME / "bin"
CUDA_LIB = CUDA_HOME / "lib"
NVCC = CUDA_BIN / "nvcc"

LOG_PATH = Path(
    "/content/ling_vllm_source_build_nvrtc_driver_fix.log"
)


def run(cmd, env=None, cwd=None, check=True):
    print("\n$", " ".join(map(str, cmd)), flush=True)

    return subprocess.run(
        [str(x) for x in cmd],
        env=env,
        cwd=cwd,
        check=check,
        text=True,
    )


print("=" * 80)
print("LING NVRTC + CUDA DRIVER FIX")
print("=" * 80)

assert REPO.exists()
assert NVCC.exists()


# ============================================================
# 1. NVRTC 13.0.88 설치
#
# NVCC 13.0.88과 동일 버전 사용
# ============================================================

print("\n[1/8] NVRTC 13.0.88 설치")

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "nvidia-cuda-nvrtc==13.0.88",
])


# ============================================================
# 2. libnvrtc.so 탐색
# ============================================================

print("\n[2/8] libnvrtc 탐색")

nvrtc_candidates = []

search_roots = [
    CUDA_HOME,
    Path("/usr/local/lib/python3.13/dist-packages/nvidia"),
]

for root in search_roots:

    if not root.exists():
        continue

    for p in root.rglob("libnvrtc.so*"):

        if p.is_file() or p.is_symlink():
            nvrtc_candidates.append(p)


nvrtc_candidates = list(
    dict.fromkeys(
        p.resolve()
        for p in nvrtc_candidates
    )
)

print("NVRTC candidates:")

for p in nvrtc_candidates:
    print(" -", p)


if not nvrtc_candidates:
    raise RuntimeError(
        "libnvrtc.so를 찾지 못했습니다."
    )


# CUDA 13 우선
nvrtc13 = [
    p
    for p in nvrtc_candidates
    if "cu13" in str(p)
]

REAL_NVRTC = (
    nvrtc13[0]
    if nvrtc13
    else nvrtc_candidates[0]
)


print("\n선택 NVRTC:")
print(REAL_NVRTC)


# ============================================================
# 3. libcuda.so.1 (NVIDIA DRIVER) 탐색
#
# Toolkit lib가 아니라 GPU driver library임
# ============================================================

print("\n[3/8] NVIDIA driver libcuda 탐색")

driver_patterns = [
    "/usr/lib64-nvidia/libcuda.so*",
    "/usr/lib/x86_64-linux-gnu/libcuda.so*",
    "/usr/local/nvidia/lib64/libcuda.so*",
    "/usr/lib/wsl/lib/libcuda.so*",
]

driver_candidates = []

import glob

for pattern in driver_patterns:

    for x in glob.glob(pattern):

        p = Path(x)

        if p.is_file() or p.is_symlink():
            driver_candidates.append(p)


driver_candidates = list(
    dict.fromkeys(
        driver_candidates
    )
)


print("Driver candidates:")

for p in driver_candidates:
    print(
        " -",
        p,
        "->",
        p.resolve()
    )


if not driver_candidates:
    raise RuntimeError(
        "NVIDIA driver libcuda.so를 찾지 못했습니다."
    )


# 실제 libcuda.so.1 우선
preferred_driver = [
    p
    for p in driver_candidates
    if p.name == "libcuda.so.1"
]

if preferred_driver:
    REAL_CUDA_DRIVER = preferred_driver[0].resolve()
else:
    REAL_CUDA_DRIVER = driver_candidates[0].resolve()


print("\n선택 CUDA DRIVER:")
print(REAL_CUDA_DRIVER)


# ============================================================
# 4. CUDA_HOME/lib에 필요한 unversioned symlink 생성
# ============================================================

print("\n[4/8] CUDA_HOME/lib 정리")

CUDA_LIB.mkdir(
    parents=True,
    exist_ok=True
)


# ---- NVRTC ----

NVRTC_LINK = CUDA_LIB / "libnvrtc.so"

if NVRTC_LINK.exists() or NVRTC_LINK.is_symlink():
    NVRTC_LINK.unlink()

NVRTC_LINK.symlink_to(
    REAL_NVRTC
)


# ---- CUDA driver ----

CUDA_DRIVER_LINK = CUDA_LIB / "libcuda.so"

if (
    CUDA_DRIVER_LINK.exists()
    or CUDA_DRIVER_LINK.is_symlink()
):
    CUDA_DRIVER_LINK.unlink()

CUDA_DRIVER_LINK.symlink_to(
    REAL_CUDA_DRIVER
)


print(
    "✅",
    NVRTC_LINK,
    "->",
    NVRTC_LINK.resolve()
)

print(
    "✅",
    CUDA_DRIVER_LINK,
    "->",
    CUDA_DRIVER_LINK.resolve()
)


assert NVRTC_LINK.exists()
assert CUDA_DRIVER_LINK.exists()


# ============================================================
# 5. 환경 설정
# ============================================================

env = os.environ.copy()

env["CUDA_HOME"] = str(CUDA_HOME)
env["CUDA_PATH"] = str(CUDA_HOME)

env["CUDAToolkit_ROOT"] = str(CUDA_HOME)

env["CUDA_TOOLKIT_ROOT_DIR"] = str(
    CUDA_HOME
)

env["CUDACXX"] = str(NVCC)

env["PATH"] = (
    f"{CUDA_BIN}:"
    + env.get("PATH", "")
)


# CMake/library search path
library_paths = [
    str(CUDA_LIB),
    "/usr/lib64-nvidia",
    "/usr/lib/x86_64-linux-gnu",
]

old_ld = env.get(
    "LD_LIBRARY_PATH",
    ""
)

env["LD_LIBRARY_PATH"] = (
    ":".join(library_paths)
    + (
        ":" + old_ld
        if old_ld
        else ""
    )
)

old_lib = env.get(
    "LIBRARY_PATH",
    ""
)

env["LIBRARY_PATH"] = (
    ":".join(library_paths)
    + (
        ":" + old_lib
        if old_lib
        else ""
    )
)


# ============================================================
# 6. CMake에 라이브러리 직접 전달
#
# CUDA_nvrtc_LIBRARY는 실제 로그에서 NOTFOUND였으므로
# 명시적으로 지정.
# driver는 libcuda.so를 toolkit lib dir에 넣어
# FindCUDAToolkit이 CUDA::cuda_driver를 생성하도록 유도.
# ============================================================

cmake_args = [

    f"-DCUDAToolkit_ROOT={CUDA_HOME}",

    f"-DCUDA_TOOLKIT_ROOT_DIR={CUDA_HOME}",

    f"-DCMAKE_CUDA_COMPILER={NVCC}",

    f"-DCUDA_CUDART_LIBRARY={CUDA_LIB / 'libcudart.so'}",

    f"-DCUDA_nvrtc_LIBRARY={NVRTC_LINK}",

    (
        "-DCMAKE_LIBRARY_PATH="
        + ";".join(library_paths)
    ),
]

env["CMAKE_ARGS"] = " ".join(
    cmake_args
)

env["MAX_JOBS"] = "2"

env["VLLM_TARGET_DEVICE"] = "cuda"


for key in [
    "VLLM_USE_PRECOMPILED",
    "VLLM_PRECOMPILED_WHEEL_COMMIT",
    "VLLM_PRECOMPILED_WHEEL_LOCATION",
    "VLLM_PRECOMPILED_WHEEL_VARIANT",
]:

    env.pop(key, None)


print("\n" + "=" * 80)
print("FINAL CUDA LIB CONFIG")
print("=" * 80)

print("CUDA_HOME :", CUDA_HOME)
print("NVCC      :", NVCC)
print("CUDART    :", CUDA_LIB / "libcudart.so")
print("NVRTC     :", NVRTC_LINK.resolve())
print("CUDA DRIVER:", CUDA_DRIVER_LINK.resolve())

print("\nCMAKE_ARGS:")
print(env["CMAKE_ARGS"])


# ============================================================
# 7. CMake standalone preflight
#
# FindCUDAToolkit이 실제 target들을 만드는지 확인
# ============================================================

print("\n[5/8] CMake CUDA imported-target preflight")

TEST_DIR = Path("/content/cuda_cmake_preflight")

if TEST_DIR.exists():
    shutil.rmtree(TEST_DIR)

TEST_DIR.mkdir()


cmake_txt = r'''
cmake_minimum_required(VERSION 3.26)

project(cuda_check LANGUAGES CXX CUDA)

find_package(CUDAToolkit REQUIRED)

message(STATUS "CUDA toolkit version=${CUDAToolkit_VERSION}")
message(STATUS "CUDA library dir=${CUDAToolkit_LIBRARY_DIR}")

if(TARGET CUDA::cudart)
  message(STATUS "CHECK_CUDART=YES")
else()
  message(FATAL_ERROR "CHECK_CUDART=NO")
endif()

if(TARGET CUDA::nvrtc)
  message(STATUS "CHECK_NVRTC=YES")
else()
  message(FATAL_ERROR "CHECK_NVRTC=NO")
endif()

if(TARGET CUDA::cuda_driver)
  message(STATUS "CHECK_CUDA_DRIVER=YES")
else()
  message(FATAL_ERROR "CHECK_CUDA_DRIVER=NO")
endif()
'''

(TEST_DIR / "CMakeLists.txt").write_text(
    cmake_txt
)


preflight = subprocess.run(

    [
        "cmake",
        "-S",
        str(TEST_DIR),
        "-B",
        str(TEST_DIR / "build"),

        f"-DCUDAToolkit_ROOT={CUDA_HOME}",
        f"-DCMAKE_CUDA_COMPILER={NVCC}",
        f"-DCMAKE_LIBRARY_PATH={';'.join(library_paths)}",
    ],

    env=env,

    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,

    text=True,
)


print(preflight.stdout)


if preflight.returncode != 0:

    raise RuntimeError(
        "CMake CUDA preflight 실패"
    )


required_markers = [
    "CHECK_CUDART=YES",
    "CHECK_NVRTC=YES",
    "CHECK_CUDA_DRIVER=YES",
]

for marker in required_markers:

    if marker not in preflight.stdout:

        raise RuntimeError(
            f"필수 target 없음: {marker}"
        )


print(
    "\n✅ CUDART / NVRTC / CUDA DRIVER target 모두 정상"
)


# ============================================================
# 8. 이전 build cache 삭제
# ============================================================

print("\n[6/8] 이전 CMake build cache 제거")

for p in [
    REPO / "build",
    REPO / "_skbuild",
]:

    if p.exists():

        shutil.rmtree(p)

        print("삭제:", p)


# CMake cache가 repo 어딘가에 남을 경우
for cache in REPO.glob(
    "**/CMakeCache.txt"
):

    # dependencies의 자체 CMake cache까지
    # 무조건 삭제하면 다시 clone할 수 있으므로
    # root build 관련 것만 삭제
    if ".deps" not in str(cache):

        try:
            shutil.rmtree(
                cache.parent
            )

            print(
                "삭제:",
                cache.parent
            )

        except Exception:
            pass


# ============================================================
# 9. vLLM Ling FULL SOURCE BUILD
# ============================================================

print("\n[7/8] FULL SOURCE BUILD")

cmd = [
    "uv",
    "pip",
    "install",
    "--system",
    "--no-build-isolation",
    "--editable",
    ".",
]


with LOG_PATH.open(
    "w",
    encoding="utf-8",
    errors="replace",
) as logfile:

    process = subprocess.Popen(

        cmd,

        cwd=REPO,

        env=env,

        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,

        text=True,
        bufsize=1,
    )

    for line in process.stdout:

        print(
            line,
            end=""
        )

        logfile.write(line)

        logfile.flush()


    return_code = process.wait()


print("\n" + "=" * 80)
print("BUILD RETURN CODE:", return_code)
print("=" * 80)


if return_code != 0:

    lines = LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    print("\nLAST 200 BUILD LOG LINES")
    print("=" * 80)

    for line in lines[-200:]:

        print(line)


    raise RuntimeError(
        "Ling source build 실패"
    )


# ============================================================
# 10. ABI 검사
# ============================================================

print("\n[8/8] MARLIN ABI CHECK")

verify_code = r'''
import torch
import inspect
import vllm
import vllm._custom_ops as ops

print("=" * 70)
print("MARLIN ABI VERIFY")
print("=" * 70)

print("vLLM:", vllm.__version__)
print("path:", ops.__file__)

wrapper = inspect.signature(
    ops.gptq_marlin_moe_repack
)

schema = (
    torch.ops._C
    .gptq_marlin_repack
    ._schemas
)

print("\nPython wrapper:")
print(wrapper)

print("\nCompiled op:")
print(schema)

if (
    "perm" in str(wrapper)
    and "perm" not in str(schema)
):
    raise RuntimeError(
        "ABI mismatch still exists"
    )

print("\n✅ MARLIN ABI MATCH")
'''


verify = subprocess.run(

    [
        sys.executable,
        "-c",
        verify_code,
    ],

    cwd=REPO,
    env=env,

    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,

    text=True,
)


print(verify.stdout)


if verify.returncode != 0:

    raise RuntimeError(
        "ABI 검사 실패"
    )


print("\n" + "=" * 80)
print("🎉 LING SOURCE BUILD COMPLETE")
print("=" * 80)

print("""
성공하면:

1. 런타임 → 세션 다시 시작
2. STEP 0 다시 실행 X
3. Ling 모델 load STEP 1 실행
""")

LING NVRTC + CUDA DRIVER FIX

[1/8] NVRTC 13.0.88 설치

$ /usr/bin/python3 -m pip install -U nvidia-cuda-nvrtc==13.0.88

[2/8] libnvrtc 탐색
NVRTC candidates:
 - /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc.so.13
 - /usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12

선택 NVRTC:
/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc.so.13

[3/8] NVIDIA driver libcuda 탐색
Driver candidates:
 - /usr/lib64-nvidia/libcuda.so -> /usr/lib64-nvidia/libcuda.so.580.82.07
 - /usr/lib64-nvidia/libcuda.so.1 -> /usr/lib64-nvidia/libcuda.so.580.82.07
 - /usr/lib64-nvidia/libcuda.so.580.82.07 -> /usr/lib64-nvidia/libcuda.so.580.82.07

선택 CUDA DRIVER:
/usr/lib64-nvidia/libcuda.so.580.82.07

[4/8] CUDA_HOME/lib 정리
✅ /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc.so -> /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc.so.13
✅ /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcuda.so -> /usr/lib64-nvidia/libcuda.so

## Step 1. Ling 3.0 Flash VL INT4 로드


In [1]:
# ============================================================
# Ling-3.0 Flash VL INT4
# STEP 1 FINAL — COLAB G4 95GB
#
# 이 셀 하나가 전부 수행:
#   1. CUDA / 패키지 환경 확인
#   2. Ling revision 확인
#   3. HF remote-code cache 복구
#   4. AutoProcessor 사전 검증
#   5. Colab stdout.fileno() 문제 패치
#   6. vLLM multiprocessing OFF
#   7. Ling INT4 모델 로드
#
# 실행 전:
#   "런타임 → 세션 다시 시작"
#
# STEP 0은 다시 실행하지 마세요.
# ============================================================


# ============================================================
# 0. vLLM IMPORT 전에 환경변수부터 설정
# ============================================================

import os

# Colab/Jupyter에서 vLLM EngineCore spawn 문제 방지
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
# 혹시 이전 값이 남아있으면 제거
os.environ.pop("VLLM_WORKER_MULTIPROC_METHOD", None)

# CUDA memory fragmentation 완화
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("=" * 80)
print("LING STEP 1 FINAL")
print("=" * 80)
print(
    "VLLM_ENABLE_V1_MULTIPROCESSING =",
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"]
)


# ============================================================
# 1. 기본 IMPORT / ENV CHECK
# ============================================================

import sys
import gc
import shutil
import importlib
import contextlib

from pathlib import Path

import torch
import torchvision
import torchaudio
import transformers
import huggingface_hub
import decord

from huggingface_hub import (
    HfApi,
    hf_hub_download,
)


MODEL_ID = "inclusionAI/Ling-3.0-flash-VL-int4"


print("\n" + "=" * 80)
print("LING ENV CHECK")
print("=" * 80)

print("Python      :", sys.version.split()[0])
print("Torch       :", torch.__version__)
print("Torch CUDA  :", torch.version.cuda)
print("TorchVision :", torchvision.__version__)
print("TorchAudio  :", torchaudio.__version__)
print("Transformers:", transformers.__version__)
print("HF Hub      :", huggingface_hub.__version__)
print("Decord      :", getattr(decord, "__version__", "import OK"))

assert torch.cuda.is_available(), (
    "GPU를 찾지 못했습니다. "
    "Colab 런타임이 G4인지 확인하세요."
)

assert torch.version.cuda == "13.0", (
    f"Torch CUDA mismatch: {torch.version.cuda}"
)

assert "cu130" in torchvision.__version__, (
    f"TorchVision mismatch: {torchvision.__version__}"
)

assert "cu130" in torchaudio.__version__, (
    f"TorchAudio mismatch: {torchaudio.__version__}"
)

assert int(huggingface_hub.__version__.split(".")[0]) < 2, (
    f"HF Hub가 너무 높음: {huggingface_hub.__version__}"
)


props = torch.cuda.get_device_properties(0)

VRAM_GIB = (
    props.total_memory
    / 2**30
)

print("GPU         :", props.name)
print("VRAM        :", f"{VRAM_GIB:.1f} GiB")

assert VRAM_GIB >= 90, (
    f"G4 95GB급 GPU 권장. 현재 {VRAM_GIB:.1f} GiB"
)

print("\n✅ CUDA / package environment 정상")


# ============================================================
# 2. 현재 Ling 모델 revision 고정
# ============================================================

print("\n" + "=" * 80)
print("HF MODEL CHECK")
print("=" * 80)

api = HfApi()

model_info = api.model_info(
    MODEL_ID
)

REVISION = model_info.sha

print("MODEL   :", MODEL_ID)
print("REVISION:", REVISION)


# ============================================================
# 3. 모델 저장소의 remote Python 코드 확인
# ============================================================

repo_files = api.list_repo_files(
    MODEL_ID,
    revision=REVISION,
)

python_files = sorted([
    f
    for f in repo_files
    if f.endswith(".py")
])

print(
    "\nRemote Python files:",
    len(python_files)
)

for f in python_files:
    print(" -", f)


required_remote_files = [
    "processing_bailingmm2.py",
    "image_processing_bailingmm2.py",
    "bailingmm_utils.py",
    "bailingmm_utils_video.py",
]

for f in required_remote_files:

    if f not in python_files:

        raise RuntimeError(
            f"모델 저장소 필수 파일 누락: {f}"
        )

print("\n✅ Ling remote-code 파일 확인 완료")


# ============================================================
# 4. Transformers dynamic-module cache 복구
#
# 이전 오류:
# bailingmm_utils_video.py가 cache에서 빠져 FileNotFoundError
#
# 이번에는 모델 저장소의 모든 *.py를 직접 복사
# ============================================================

print("\n" + "=" * 80)
print("REBUILD TRANSFORMERS REMOTE-CODE CACHE")
print("=" * 80)


DYNAMIC_ROOT = Path(
    "/root/.cache/huggingface/modules/"
    "transformers_modules"
)

ORG_DIR = (
    DYNAMIC_ROOT
    / "inclusionAI"
)

MODULE_NAME = (
    "Ling_hyphen_3_dot_0_hyphen_flash"
    "_hyphen_VL_hyphen_int4"
)

MODEL_MODULE_ROOT = (
    ORG_DIR
    / MODULE_NAME
)

TARGET = (
    MODEL_MODULE_ROOT
    / REVISION
)


# 기존 revision cache가 깨졌을 수 있으므로
# revision 폴더만 깨끗하게 재생성
if TARGET.exists():

    shutil.rmtree(
        TARGET
    )

TARGET.mkdir(
    parents=True,
    exist_ok=True
)


# package __init__.py 생성
for directory in [
    DYNAMIC_ROOT,
    ORG_DIR,
    MODEL_MODULE_ROOT,
    TARGET,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

    init_file = (
        directory
        / "__init__.py"
    )

    if not init_file.exists():
        init_file.touch()


# 모든 remote-code Python 파일 복사
for filename in python_files:

    source = Path(
        hf_hub_download(
            repo_id=MODEL_ID,
            filename=filename,
            revision=REVISION,
        )
    )

    destination = (
        TARGET
        / filename
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        source,
        destination
    )

    # subfolder가 존재할 경우 package 처리
    parent = destination.parent

    while (
        parent != TARGET
        and TARGET in parent.parents
    ):

        init_file = (
            parent
            / "__init__.py"
        )

        if not init_file.exists():
            init_file.touch()

        parent = parent.parent

    print(
        "✅",
        filename
    )


# 실제 문제 파일 체크
VIDEO_UTIL_FILE = (
    TARGET
    / "bailingmm_utils_video.py"
)

assert VIDEO_UTIL_FILE.is_file(), (
    "bailingmm_utils_video.py 복구 실패"
)

print(
    "\n✅ bailingmm_utils_video.py 확인:",
    VIDEO_UTIL_FILE
)

importlib.invalidate_caches()


# ============================================================
# 5. Transformers dynamic module 초기화
# ============================================================

from transformers.dynamic_module_utils import (
    init_hf_modules,
)

init_hf_modules()

importlib.invalidate_caches()


# ============================================================
# 6. Processor preflight
#
# 모델 GPU 로드 전에 Processor부터 확인
# ============================================================

print("\n" + "=" * 80)
print("PROCESSOR PREFLIGHT")
print("=" * 80)

from transformers import AutoProcessor

processor_check = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    trust_remote_code=True,
)

print(
    "Processor     :",
    type(processor_check).__name__
)

print(
    "ImageProcessor:",
    type(
        processor_check.image_processor
    ).__name__
)

print("\n✅ PROCESSOR SUCCESS")

del processor_check

gc.collect()


# ============================================================
# 7. vLLM import
# ============================================================

print("\n" + "=" * 80)
print("VLLM IMPORT")
print("=" * 80)

import vllm

print(
    "vLLM:",
    vllm.__version__
)


# ============================================================
# 8. Colab/Jupyter stdout.fileno() HOTFIX
#
# vLLM fork 내부:
#
#   with suppress_stdout():
#       torch.distributed.new_group(...)
#
# 그런데 Colab stdout에는 fileno()가 없어
# UnsupportedOperation: fileno 발생.
#
# suppress_stdout은 출력 억제용이므로 notebook에서는 no-op.
# ============================================================

@contextlib.contextmanager
def notebook_safe_suppress_stdout():
    yield


import vllm.utils.system_utils as system_utils
import vllm.distributed.parallel_state as parallel_state


# 원본 함수 교체
system_utils.suppress_stdout = (
    notebook_safe_suppress_stdout
)

parallel_state.suppress_stdout = (
    notebook_safe_suppress_stdout
)


# fork에 따라 다른 곳에서 직접 import했을 수도 있으므로
# 이미 로드된 모든 vllm module의 suppress_stdout도 교체
patched_modules = []

for module_name, module in list(
    sys.modules.items()
):

    if not (
        module_name == "vllm"
        or module_name.startswith("vllm.")
    ):
        continue

    if module is None:
        continue

    if hasattr(
        module,
        "suppress_stdout"
    ):

        try:

            setattr(
                module,
                "suppress_stdout",
                notebook_safe_suppress_stdout
            )

            patched_modules.append(
                module_name
            )

        except Exception:
            pass


# 혹시 아직 import되지 않은 distributed.utils도 처리
try:

    import vllm.distributed.utils as distributed_utils

    distributed_utils.suppress_stdout = (
        notebook_safe_suppress_stdout
    )

except Exception:
    pass


print(
    "suppress_stdout patched modules:",
    len(patched_modules)
)

print(
    "parallel_state.suppress_stdout:",
    parallel_state.suppress_stdout
)

assert (
    parallel_state.suppress_stdout
    is notebook_safe_suppress_stdout
)

print("\n✅ Colab stdout.fileno() patch 성공")


# ============================================================
# 9. LLM import
# ============================================================

from vllm import LLM

print("✅ from vllm import LLM 성공")

print(
    "V1 multiprocessing:",
    os.environ[
        "VLLM_ENABLE_V1_MULTIPROCESSING"
    ]
)


# ============================================================
# 10. GPU CLEAN
# ============================================================

gc.collect()

torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()

print("\n" + "=" * 80)
print("GPU BEFORE LOAD")
print("=" * 80)

print(
    "Free :",
    f"{free / 2**30:.2f} GiB"
)

print(
    "Total:",
    f"{total / 2**30:.2f} GiB"
)


# ============================================================
# 11. Ling INT4 모델 로드
# ============================================================

print("\n" + "=" * 80)
print("LING MODEL LOAD — IN-PROCESS MODE")
print("=" * 80)

print("MODEL   :", MODEL_ID)
print("REVISION:", REVISION)
print("GPU     :", props.name)
print("VRAM    :", f"{VRAM_GIB:.1f} GiB")


llm = LLM(

    model=MODEL_ID,

    revision=REVISION,

    trust_remote_code=True,

    # G4 single GPU
    tensor_parallel_size=1,

    # INT4 weight + BF16 compute
    dtype="bfloat16",

    # VQA에서 우선 충분
    max_model_len=4096,

    # 문제 한 개씩
    max_num_seqs=1,

    # 첫 로딩 안정성 우선
    gpu_memory_utilization=0.90,

    # torch.compile / cudagraph 회피
    enforce_eager=True,

    # 문제당 이미지 한 장
    limit_mm_per_prompt={
        "image": 1
    },

    seed=42,
)


# ============================================================
# 12. SUCCESS / MEMORY
# ============================================================

print("\n" + "=" * 80)
print("🎉 LING-3.0-FLASH-VL INT4 LOAD SUCCESS")
print("=" * 80)

free, total = torch.cuda.mem_get_info()

print(
    "CUDA free      :",
    f"{free / 2**30:.2f} GiB"
)

print(
    "CUDA total     :",
    f"{total / 2**30:.2f} GiB"
)

print(
    "Torch allocated:",
    f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB"
)

print(
    "Torch reserved :",
    f"{torch.cuda.memory_reserved() / 2**30:.2f} GiB"
)

print("=" * 80)

LING STEP 1 FINAL
VLLM_ENABLE_V1_MULTIPROCESSING = 0

LING ENV CHECK
Python      : 3.13.15
Torch       : 2.13.0+cu130
Torch CUDA  : 13.0
TorchVision : 0.28.0+cu130
TorchAudio  : 2.11.0+cu130
Transformers: 5.17.0
HF Hub      : 1.33.0
Decord      : 3.4.0
GPU         : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM        : 95.0 GiB

✅ CUDA / package environment 정상

HF MODEL CHECK
MODEL   : inclusionAI/Ling-3.0-flash-VL-int4
REVISION: 94759a4b454498d91144582e26c1341ba1acb5d4

Remote Python files: 10
 - bailingmm_utils.py
 - bailingmm_utils_video.py
 - chat_format.py
 - configuration_bailing_moe_v3_vl.py
 - configuration_bailingmm.py
 - image_processing_bailingmm2.py
 - modeling_bailingmm.py
 - modeling_utils.py
 - processing_bailingmm2.py
 - tokenization_bailing.py

✅ Ling remote-code 파일 확인 완료

REBUILD TRANSFORMERS REMOTE-CODE CACHE
✅ bailingmm_utils.py
✅ bailingmm_utils_video.py
✅ chat_format.py
✅ configuration_bailing_moe_v3_vl.py
✅ configuration_bailingmm.py
✅ image_processing_baili

[transformers] `BailingMM2Processor` defines `image_processor_class = 'AutoImageProcessor'`, which is deprecated. Register the correct mapping in `AutoImageProcessor` instead.


Processor     : BailingMM2Processor
ImageProcessor: BailingMM2ImageProcessor

✅ PROCESSOR SUCCESS

VLLM IMPORT
vLLM: 0.1.dev1+ge2e5751a5.d20260926
suppress_stdout patched modules: 4
parallel_state.suppress_stdout: <function notebook_safe_suppress_stdout at 0x7ae99e473740>

✅ Colab stdout.fileno() patch 성공
✅ from vllm import LLM 성공
V1 multiprocessing: 0

GPU BEFORE LOAD
Free : 94.43 GiB
Total: 94.97 GiB

LING MODEL LOAD — IN-PROCESS MODE
MODEL   : inclusionAI/Ling-3.0-flash-VL-int4
REVISION: 94759a4b454498d91144582e26c1341ba1acb5d4
GPU     : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM    : 95.0 GiB
INFO 09-27 01:07:23 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'seed': 42, 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 1, 'disable_log_stats': True, 'revision': '94759a4b454498d91144582e26c1341ba1acb5d4', 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 1}, 'model': 'inclusionAI/Ling-3.0-flash-VL-int4'}
WARNING 

INFO 09-27 01:07:30 [parallel_state.py:1798] world_size=1 rank=0 local_rank=0 distributed_init_method=file:///tmp/vllm_dist_417d15d500994523b199eca4debbf33f backend=nccl
INFO 09-27 01:07:30 [parallel_state.py:2142] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0, EPLB rank N/A
INFO 09-27 01:07:30 [gpu_worker.py:433] Using V2 Model Runner
INFO 09-27 01:07:31 [model_runner.py:383] Loading model from scratch...
INFO 09-27 01:07:31 [cuda.py:551] Using backend AttentionBackendEnum.FLASH_ATTN for vit attention
INFO 09-27 01:07:31 [mm_encoder_attention.py:372] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
INFO 09-27 01:07:31 [kernel.py:372] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 09-27 01:07:31 [compressed_tensors_moe.py:132] Using CompressedTensorsWNA16MoEMethod
INFO 09-27 01:07:31 [int_wna16.py:297] Using 'MARLIN' WNA16 Mo

Loading safetensors checkpoint shards:   0% Completed | 0/64 [00:00<?, ?it/s]


...

INFO 09-27 01:08:07 [kernel_warmup.py:282] Running FlashInfer autotune with 4096 tokens.


2026-09-27 01:08:07,683 - INFO - autotuner.py:995 - flashinfer.jit: [Autotuner]: Autotuning process ends
2026-09-27 01:08:07,686 - INFO - autotuner.py:2652 - flashinfer.jit: [Autotuner]: Saved 0 configs to /root/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.18/e2e346bf35281bf645dede0ccadcc3a68e6d3ea957a6ef9bcc2814afc7327b46/autotune_configs.json (0 new, 0 from previous config)


INFO 09-27 01:08:07 [gpu_worker.py:864] Free memory on device (94.43/94.97 GiB) on startup. Desired GPU memory utilization is (0.9, 85.47 GiB). Actual usage is 72.23 GiB for consumed memory (weights + non-torch), 0.65 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=13361536615` (12.44 GiB) to fit into requested memory, or `--kv-cache-memory=22973642752` (21.4 GiB) to fully utilize gpu memory. Current kv cache memory in use is 12.59 GiB.


/usr/local/lib/python3.13/dist-packages/triton/language/core.py:2284: UserWarning: tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.
  warn("tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.")


INFO 09-27 01:08:10 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
INFO 09-27 01:08:11 [torch_utils.py:280] Reducing Torch threads from 24 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
INFO 09-27 01:08:11 [core.py:368] init engine (profile, create kv cache, warmup model) took 6.93 s
INFO 09-27 01:08:14 [hf.py:589] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.
INFO 09-27 01:08:20 [base.py:244] Multi-modal warmup completed in 6.301s
INFO 09-27 01:08:20 [base.py:244] Readonly multi-modal warmup completed in 0.143s

🎉 LING-3.0-FLASH-VL INT4 LOAD SUCCESS
CUDA free      : 9.14 GiB
CUDA total     : 94.97 GiB
Torch allocated: 84.08 GiB
Torch reserved : 85.12 GiB


## Step 2. Test 데이터와 VQA 프롬프트 준비


In [2]:
# ============================================================
# STEP 2. DATA + PROMPT
# - /content/drive/MyDrive/data.zip (비밀번호 없는 일반 ZIP)
# - test.csv / test/ / sample_submission.csv 자동 확인
# - OCR / DINO / 외부 API 없음
# ============================================================

from pathlib import Path
import zipfile, shutil, os, time, inspect
import pandas as pd
from PIL import Image, ImageOps
from google.colab import drive
from vllm import SamplingParams
from vllm.sampling_params import StructuredOutputsParams

drive.mount("/content/drive")

DATA_ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/data.zip"),
    Path("/content/drive/MyDrive/mori/data.zip"),
]
DATA_ZIP = next((p for p in DATA_ZIP_CANDIDATES if p.is_file()), None)
if DATA_ZIP is None:
    raise FileNotFoundError(f"data.zip을 찾지 못했습니다: {DATA_ZIP_CANDIDATES}")

DATA_ROOT = Path("/content/vqa_data_plain")

def prepare_plain_zip(zip_path, out_dir):
    out_dir = Path(out_dir)

    # 이미 준비된 경우 그대로 재사용
    roots = []
    if out_dir.exists():
        roots = sorted({
            p.parent for p in out_dir.rglob("test.csv")
            if (p.parent / "sample_submission.csv").is_file()
            and (p.parent / "test").is_dir()
        })
    if len(roots) == 1:
        print("기존 데이터 재사용:", roots[0])
        return roots[0]

    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("일반 ZIP 압축 해제:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        encrypted = [x.filename for x in zf.infolist() if x.flag_bits & 1]
        if encrypted:
            raise RuntimeError(f"암호화된 파일이 있습니다: {encrypted[:5]}")
        zf.extractall(out_dir)

    roots = sorted({
        p.parent for p in out_dir.rglob("test.csv")
        if (p.parent / "sample_submission.csv").is_file()
        and (p.parent / "test").is_dir()
    })
    if len(roots) != 1:
        raise RuntimeError(f"test 데이터 root를 하나 찾지 못했습니다: {roots}")
    return roots[0]

DATA_ROOT = prepare_plain_zip(DATA_ZIP, DATA_ROOT)
TEST_CSV = DATA_ROOT / "test.csv"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"

test_df = pd.read_csv(TEST_CSV)
sample_submission = pd.read_csv(SAMPLE_CSV, dtype={"id": str})

required = ["id","path","question","a","b","c","d"]
missing = [c for c in required if c not in test_df.columns]
assert not missing, f"test.csv 컬럼 누락: {missing}"
assert len(test_df) == len(sample_submission), (len(test_df), len(sample_submission))
assert test_df["id"].astype(str).tolist() == sample_submission["id"].astype(str).tolist()

exists = test_df["path"].map(lambda p: (DATA_ROOT / str(p)).is_file())
assert exists.all(), f"누락 이미지 {int((~exists).sum())}개"

SYSTEM_PROMPT = (
    "당신은 이미지 기반 4지선다 VQA 전문가입니다.\n"
    "이미지를 가장 중요한 근거로 사용하고 질문과 선택지를 신중하게 비교하세요.\n"
    "최종 답은 반드시 a, b, c, d 중 소문자 한 글자로만 출력하세요."
)

def build_prompt(row):
    return (
        f"질문: {row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        "정답:"
    )

choice_constraint = StructuredOutputsParams(choice=["a","b","c","d"])
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=2,
    structured_outputs=choice_constraint,
)

SUPPORTS_USE_TQDM = "use_tqdm" in inspect.signature(llm.chat).parameters

def load_image(path, max_side=None):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        if max_side is not None:
            im.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
        return im.copy()

def infer_one(row, max_side=None):
    image_path = DATA_ROOT / str(row["path"])
    image = load_image(image_path, max_side=max_side)

    messages = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":[
            {"type":"image_pil","image_pil":image},
            {"type":"text","text":build_prompt(row)},
        ]},
    ]

    kwargs = dict(
        messages=messages,
        sampling_params=sampling_params,
        chat_template_kwargs={"enable_thinking": False},
    )
    if SUPPORTS_USE_TQDM:
        kwargs["use_tqdm"] = False

    outputs = llm.chat(**kwargs)
    pred = outputs[0].outputs[0].text.strip().lower()

    if pred not in {"a","b","c","d"}:
        raise ValueError(f"비정상 prediction: {pred!r}")
    return pred

print("=" * 72)
print("DATA READY")
print("ROOT :", DATA_ROOT)
print("TEST :", len(test_df))
print("IMAGE:", int(exists.sum()))
print("=" * 72)

display(test_df.head(3))


Mounted at /content/drive
일반 ZIP 압축 해제: /content/drive/MyDrive/data.zip
DATA READY
ROOT : /content/vqa_data_plain
TEST : 6714
IMAGE: 6714


,id,path,question,a,b,c,d
0,test_0001.jpg,test/test_0001.jpg,이 사진 속 간판 중에서 '머리염색 체험방'이 적혀 있는 간판의 전화번호는 무엇인가요?,010-1234-5678,722-7510,123-4567,732-3258
1,test_0002.jpg,test/test_0002.jpg,이 식당 간판에 적힌 전문점의 종류는 무엇인가요?,중식 전문점,해산물 전문점,소고기숯불구이 전문점,한식 전문점
2,test_0003.jpg,test/test_0003.jpg,이 사진에 보이는 IT대학 발전기금 현황판에서 IT대학 50주년 후원금 중 10억원...,SK hynix,LG,삼성전자,세 가지 모두


## Step 3. 3문제 smoke


In [3]:
# ============================================================
# STEP 3. 3개 TEST SMOKE
# - 정답 없음. 모델/이미지/프롬프트 경로가 정상인지 확인용.
# ============================================================

for i in range(3):
    row = test_df.iloc[i]
    t0 = time.time()
    pred = infer_one(row)
    print(
        f"{i}: {row['id']} -> {pred} "
        f"({time.time()-t0:.2f}s) | {row['question']}"
    )

print("✅ smoke 완료. 위 3개가 a/b/c/d로 나오면 STEP 4 실행.")


WARNING 09-27 01:12:18 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _bilinear_pos_embed_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
0: test_0001.jpg -> b (1.00s) | 이 사진 속 간판 중에서 '머리염색 체험방'이 적혀 있는 간판의 전화번호는 무엇인가요?
1: test_0002.jpg -> c (0.18s) | 이 식당 간판에 적힌 전문점의 종류는 무엇인가요?
2: test_0003.jpg -> d (0.17s) | 이 사진에 보이는 IT대학 발전기금 현황판에서 IT대학 50주년 후원금 중 10억원 이상을 기부한 기업은 어떤 것인가요?
✅ smoke 완료. 위 3개가 a/b/c/d로 나오면 STEP 4 실행.


In [4]:
# ============================================================
# STEP 3.5. FIXED LABELED 128 BENCHMARK
#
# 목적:
# - GLM 때 사용한 정확히 동일한 128문항 재사용
# - Ling zero-shot 정확도 확인
# - 결과 보고 STEP 4 전체 TEST 진행 여부 결정
#
# 현재 llm / infer_one / DATA_ROOT 그대로 사용
# ============================================================

import time
import gc
from pathlib import Path

import pandas as pd
import torch


# ============================================================
# 1. GLM 때 만든 공용 128 subset
# ============================================================

COMMON_BENCH_DIR = Path(
    "/content/drive/MyDrive/cleaning_data/model_benchmark"
)

VALID_SUBSET_CSV = (
    COMMON_BENCH_DIR
    / "benchmark_labeled_128.csv"
)

CLEAN_CSV = Path(
    "/content/drive/MyDrive/cleaning_data/train_combined_clean.csv"
)

OUT_DIR_128 = Path(
    "/content/drive/MyDrive/cleaning_data/ling3_vl_int4_eval"
)

OUT_DIR_128.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_128_CSV = (
    OUT_DIR_128
    / "ling3_vl_int4_labeled_128_result.csv"
)


# ============================================================
# 2. 정확히 동일한 fixed subset 가져오기
# ============================================================

if VALID_SUBSET_CSV.exists():

    valid_128 = pd.read_csv(
        VALID_SUBSET_CSV
    )

    print(
        "✅ GLM 때 사용한 fixed 128 재사용:"
    )
    print(
        VALID_SUBSET_CSV
    )

else:

    # 혹시 공용 subset 파일이 없어졌다면
    # GLM 노트북과 동일한 방식으로 재생성
    assert CLEAN_CSV.exists(), (
        f"clean dataset 없음: {CLEAN_CSV}"
    )

    labeled_df = pd.read_csv(
        CLEAN_CSV
    )

    valid_128 = (
        labeled_df
        .sample(
            n=min(128, len(labeled_df)),
            random_state=42,
        )
        .copy()
        .reset_index()
        .rename(
            columns={
                "index": "source_idx"
            }
        )
    )

    COMMON_BENCH_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    valid_128.to_csv(
        VALID_SUBSET_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "✅ GLM과 동일 규칙으로 fixed 128 재생성:"
    )
    print(
        VALID_SUBSET_CSV
    )


required_cols = {
    "id",
    "path",
    "question",
    "a",
    "b",
    "c",
    "d",
    "answer",
}

assert required_cols.issubset(
    valid_128.columns
), (
    "128 subset 컬럼 부족: "
    + str(
        required_cols
        - set(valid_128.columns)
    )
)

assert len(valid_128) == 128, (
    f"128개가 아님: {len(valid_128)}"
)


print("\nRows:", len(valid_128))

print(
    "GT distribution:"
)

print(
    valid_128["answer"]
    .astype(str)
    .str.lower()
    .value_counts()
    .sort_index()
)


# ============================================================
# 3. 이미지 경로 resolver
#
# 현재 Ling DATA_ROOT
# 과거 GLM / 기타 추출 경로 모두 대응
# ============================================================

IMAGE_ROOT_CANDIDATES = [

    DATA_ROOT,

    Path(
        "/content/vqa_data_plain"
    ),

    Path(
        "/content/ssafy_data"
    ),

    Path(
        "/content/agnes_test_data_plain"
    ),
]


IMAGE_ROOT_CANDIDATES = [
    p
    for p in IMAGE_ROOT_CANDIDATES
    if Path(p).exists()
]


def resolve_benchmark_image(
    raw_path
):

    raw_path = Path(
        str(raw_path)
    )

    # 이미 절대경로인 경우
    if (
        raw_path.is_absolute()
        and raw_path.is_file()
    ):
        return raw_path


    # 알려진 root 기준
    for root in IMAGE_ROOT_CANDIDATES:

        candidate = (
            Path(root)
            / raw_path
        )

        if candidate.is_file():
            return candidate


    # path의 앞 디렉터리 구조가 달라진 경우
    # 파일명으로 마지막 fallback
    basename = raw_path.name

    for root in IMAGE_ROOT_CANDIDATES:

        matches = list(
            Path(root).rglob(
                basename
            )
        )

        matches = [
            x
            for x in matches
            if x.is_file()
        ]

        if len(matches) == 1:
            return matches[0]

        if len(matches) > 1:

            # 동일 basename 여러 개면
            # 원래 path suffix와 최대한 맞는 것
            raw_parts = raw_path.parts

            for m in matches:

                m_str = str(m)

                suffix = str(
                    raw_path
                )

                if m_str.endswith(
                    suffix
                ):
                    return m


    raise FileNotFoundError(
        f"이미지를 찾지 못함: {raw_path}"
    )


# ============================================================
# 4. 먼저 이미지 128개 경로 검증
# ============================================================

resolved_paths = []
missing = []

for i, row in valid_128.iterrows():

    try:

        p = resolve_benchmark_image(
            row["path"]
        )

        resolved_paths.append(
            str(p)
        )

    except Exception as e:

        resolved_paths.append(
            ""
        )

        missing.append(
            (
                i,
                row["id"],
                row["path"],
                repr(e),
            )
        )


print(
    "\nIMAGE CHECK:",
    len(valid_128) - len(missing),
    "/",
    len(valid_128),
)


if missing:

    print("\n❌ 누락 이미지 예시:")

    for x in missing[:10]:
        print(x)

    raise RuntimeError(
        f"128개 중 이미지 {len(missing)}개를 찾지 못했습니다."
    )


valid_128 = valid_128.copy()

valid_128[
    "_resolved_path"
] = resolved_paths


print(
    "✅ 128개 이미지 전부 확인"
)


# ============================================================
# 5. Ling 128 inference
# ============================================================

records = []

bench_start = time.time()


print("\n" + "=" * 78)
print("LING-3.0-FLASH-VL INT4 — FIXED LABELED 128")
print("=" * 78)


for i, row in valid_128.iterrows():

    t0 = time.time()

    pred = ""
    status = "error"
    error = ""
    used_max_side = "original"


    # infer_one()이 DATA_ROOT / path 형태로 읽으므로
    # absolute path를 넣은 row 복사본 사용
    row_for_infer = row.copy()

    row_for_infer[
        "path"
    ] = row[
        "_resolved_path"
    ]


    # 원본 우선
    # 오류일 때만 축소 재시도
    for max_side in [
        None,
        1600,
        1280,
        1024,
    ]:

        try:

            pred = infer_one(
                row_for_infer,
                max_side=max_side,
            )

            status = "ok"

            used_max_side = (
                "original"
                if max_side is None
                else str(max_side)
            )

            error = ""

            break

        except Exception as e:

            error = repr(e)

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


    elapsed = (
        time.time()
        - t0
    )

    gt = (
        str(
            row["answer"]
        )
        .strip()
        .lower()
    )


    records.append({

        "bench_idx":
            int(i),

        "source_idx":
            row.get(
                "source_idx",
                ""
            ),

        "id":
            str(
                row["id"]
            ),

        "path":
            row["path"],

        "resolved_path":
            row[
                "_resolved_path"
            ],

        "gt_answer":
            gt,

        "ling_answer":
            pred,

        "correct":
            pred == gt,

        "status":
            status,

        "elapsed_sec":
            round(
                elapsed,
                4
            ),

        "used_max_side":
            used_max_side,

        "error":
            error,
    })


    # 8개마다 중간 저장
    if (
        (i + 1) % 8 == 0
        or (i + 1) == len(valid_128)
    ):

        tmp = pd.DataFrame(
            records
        )

        tmp.to_csv(
            RESULT_128_CSV,
            index=False,
            encoding="utf-8-sig",
        )

        ok_now = (
            tmp[
                "status"
            ].eq("ok")
        )

        acc_now = (
            tmp.loc[
                ok_now,
                "correct"
            ].mean()
            if ok_now.any()
            else 0.0
        )

        avg_now = (
            tmp.loc[
                ok_now,
                "elapsed_sec"
            ].mean()
            if ok_now.any()
            else 0.0
        )

        print(
            f"[{i+1:03d}/128] "
            f"ACC={acc_now:.4f} | "
            f"AVG={avg_now:.3f}s"
        )


# ============================================================
# 6. 결과 요약
# ============================================================

result_128 = pd.DataFrame(
    records
)

ok = (
    result_128[
        "status"
    ].eq("ok")
)

valid_ok = (
    result_128[
        ok
    ].copy()
)


total_correct = int(
    valid_ok[
        "correct"
    ].sum()
)

total_wrong = (
    len(valid_ok)
    - total_correct
)

accuracy = (
    valid_ok[
        "correct"
    ].mean()
    if len(valid_ok)
    else 0.0
)


print("\n" + "=" * 78)
print("LING FIXED LABELED 128 RESULT")
print("=" * 78)

print(
    "Rows       :",
    len(result_128)
)

print(
    "OK         :",
    int(ok.sum())
)

print(
    "Errors     :",
    int((~ok).sum())
)

print(
    "Accuracy   :",
    f"{accuracy:.6f}"
)

print(
    "Correct    :",
    total_correct
)

print(
    "Wrong      :",
    total_wrong
)

print(
    "Avg sec/img:",
    f"{valid_ok['elapsed_sec'].mean():.3f}"
)

print(
    "Med sec/img:",
    f"{valid_ok['elapsed_sec'].median():.3f}"
)

print(
    "Wall time  :",
    f"{time.time() - bench_start:.1f}s"
)

print(
    "Saved      :",
    RESULT_128_CSV
)


# ============================================================
# 7. GT별 정확도
# ============================================================

print("\nAccuracy by GT answer:")

by_gt = (
    valid_ok
    .groupby(
        "gt_answer"
    )["correct"]
    .agg(
        [
            "count",
            "sum",
            "mean",
        ]
    )
)

print(
    by_gt
)


print(
    "\nPrediction distribution:"
)

print(
    valid_ok[
        "ling_answer"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 8. GLM 기준과 단순 비교
# ============================================================

GLM_CORRECT = 97
GLM_TOTAL = 128
GLM_ACC = (
    GLM_CORRECT
    / GLM_TOTAL
)

print("\n" + "=" * 78)
print("REFERENCE")
print("=" * 78)

print(
    f"GLM : "
    f"{GLM_CORRECT}/{GLM_TOTAL} "
    f"= {GLM_ACC:.6f}"
)

print(
    f"Ling: "
    f"{total_correct}/{len(valid_ok)} "
    f"= {accuracy:.6f}"
)

print(
    f"Δ accuracy vs GLM: "
    f"{accuracy - GLM_ACC:+.6f}"
)

print("=" * 78)


# 오답 일부 확인용
wrong_df = (
    valid_ok[
        ~valid_ok["correct"]
    ][
        [
            "id",
            "gt_answer",
            "ling_answer",
            "elapsed_sec",
        ]
    ]
)

print(
    "\nWrong samples:",
    len(wrong_df)
)

display(
    wrong_df.head(20)
)

✅ GLM과 동일 규칙으로 fixed 128 재생성:
/content/drive/MyDrive/cleaning_data/model_benchmark/benchmark_labeled_128.csv

Rows: 128
GT distribution:
answer
a    39
b    26
c    33
d    30
Name: count, dtype: int64

IMAGE CHECK: 128 / 128
✅ 128개 이미지 전부 확인

LING-3.0-FLASH-VL INT4 — FIXED LABELED 128
WARNING 09-27 01:15:34 [jit_monitor.py:141] Triton kernel JIT compilation during inference: rotary_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 09-27 01:15:35 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _fwd_grouped_kernel_stage1. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 09-27 01:15:35 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _fwd_kernel_stage2. This causes a latency spike; consider extending warmup to cover this shape/config.
[008/128] ACC=0.8750 | AVG=0.251s
[016/128] ACC=0.9375 | AVG=0.212s
[024/128] ACC=0.8750 | AVG=0.203s
[032/128] ACC=0.8750 | 

,id,gt_answer,ling_answer,elapsed_sec
2,dev_2232.jpg,d,c,0.1681
18,train_0300.jpg,a,c,0.1697
20,train_3800.jpg,b,c,0.1716
31,dev_0410.jpg,b,d,0.1699
32,train_5393.jpg,a,d,0.1694
56,dev_1553.jpg,a,c,0.1687
60,train_3446.jpg,b,d,0.1693
83,train_5463.jpg,c,a,0.1703


In [5]:
# ============================================================
# STEP 3.6 — THINKING A/B TEST
#
# Direct  : 기존 infer_one()
# Thinking: infer_one_thinking()
#
# 동일 fixed 128에 비교
# - direct accuracy
# - thinking accuracy
# - rescued / harmed
# - disagreement
# ============================================================

import re
import time
import gc
from pathlib import Path

import pandas as pd
import torch
from vllm import SamplingParams


# ============================================================
# 1. Thinking용 SamplingParams
#
# structured_outputs를 일부러 사용하지 않음
# reasoning을 생성한 뒤 최종 답을 파싱
# ============================================================

thinking_sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=256,
)


# ============================================================
# 2. Thinking용 prompt
#
# reasoning은 자유롭게 하되,
# 마지막 줄을 FINAL: x 로 강하게 유도
# ============================================================

THINKING_SYSTEM_PROMPT = (
    "당신은 이미지 기반 4지선다 VQA 전문가입니다.\n"
    "이미지를 가장 중요한 근거로 사용하고 질문과 선택지를 신중하게 분석하세요.\n"
    "필요하면 이미지 속 글자, 숫자, 위치, 간판, 표, 색상 등을 단계적으로 확인하세요.\n"
    "충분히 추론한 뒤 마지막에는 반드시 다음 형식으로 답하세요:\n"
    "FINAL: a\n"
    "FINAL의 답은 반드시 a, b, c, d 중 하나여야 합니다."
)


def build_thinking_prompt(row):
    return (
        f"질문: {row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        "이미지를 확인하고 충분히 추론한 뒤 정답을 결정하세요."
    )


# ============================================================
# 3. 최종 답 parser
# ============================================================

def parse_thinking_answer(text):
    text = str(text).strip()

    # 가장 우선: FINAL: a
    matches = re.findall(
        r"FINAL\s*[:：]\s*([abcd])\b",
        text,
        flags=re.IGNORECASE,
    )

    if matches:
        return matches[-1].lower()


    # 한국어 형태 fallback
    patterns = [
        r"최종\s*답(?:변)?\s*[:：]?\s*\(?([abcd])\)?",
        r"정답\s*(?:은|:|：)?\s*\(?([abcd])\)?",
        r"answer\s*[:：]\s*\(?([abcd])\)?",
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        if matches:
            return matches[-1].lower()


    # 마지막 줄 쪽에서 단독 a/b/c/d 찾기
    tail = text[-300:]

    matches = re.findall(
        r"(?:^|\s|\()([abcd])(?:\)|\.|,|\s|$)",
        tail,
        flags=re.IGNORECASE,
    )

    if matches:
        return matches[-1].lower()


    raise ValueError(
        f"Thinking 답 파싱 실패: {text[-500:]!r}"
    )


# ============================================================
# 4. Thinking inference
# ============================================================

def infer_one_thinking(
    row,
    max_side=None,
    return_raw=False,
):

    image_path = (
        DATA_ROOT
        / str(row["path"])
    )

    image = load_image(
        image_path,
        max_side=max_side,
    )

    messages = [
        {
            "role": "system",
            "content": THINKING_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_pil",
                    "image_pil": image,
                },
                {
                    "type": "text",
                    "text": build_thinking_prompt(row),
                },
            ],
        },
    ]


    kwargs = dict(
        messages=messages,
        sampling_params=thinking_sampling_params,

        # ★ 핵심
        chat_template_kwargs={
            "enable_thinking": True
        },
    )


    if SUPPORTS_USE_TQDM:
        kwargs["use_tqdm"] = False


    outputs = llm.chat(
        **kwargs
    )

    raw = (
        outputs[0]
        .outputs[0]
        .text
        .strip()
    )

    pred = parse_thinking_answer(
        raw
    )


    if pred not in {
        "a", "b", "c", "d"
    }:

        raise ValueError(
            f"비정상 prediction: {pred!r}\nRAW={raw!r}"
        )


    if return_raw:
        return pred, raw

    return pred


print("✅ infer_one_thinking 준비 완료")


# ============================================================
# 5. 기존 fixed 128 불러오기
# ============================================================

BENCH_CSV = Path(
    "/content/drive/MyDrive/cleaning_data/"
    "model_benchmark/benchmark_labeled_128.csv"
)

assert BENCH_CSV.exists(), (
    f"benchmark 없음: {BENCH_CSV}"
)

bench = pd.read_csv(
    BENCH_CSV
)

assert len(bench) == 128


# ============================================================
# 6. 기존 Direct 결과도 불러오기
# ============================================================

DIRECT_RESULT = Path(
    "/content/drive/MyDrive/cleaning_data/"
    "ling3_vl_int4_eval/"
    "ling3_vl_int4_labeled_128_result.csv"
)

assert DIRECT_RESULT.exists(), (
    f"Direct 128 결과 없음: {DIRECT_RESULT}"
)

direct_df = pd.read_csv(
    DIRECT_RESULT,
    dtype={"id": str},
)


direct_map = dict(
    zip(
        direct_df["id"].astype(str),
        direct_df["ling_answer"]
        .astype(str)
        .str.lower(),
    )
)


# ============================================================
# 7. benchmark 이미지 resolver
# ============================================================

ROOTS = [
    DATA_ROOT,
    Path("/content/vqa_data_plain"),
]


def resolve_img(raw):

    p = Path(
        str(raw)
    )

    if (
        p.is_absolute()
        and p.is_file()
    ):
        return p


    for root in ROOTS:

        q = Path(root) / p

        if q.is_file():
            return q


    name = p.name

    for root in ROOTS:

        matches = list(
            Path(root).rglob(
                name
            )
        )

        if matches:
            return matches[0]


    raise FileNotFoundError(
        raw
    )


# ============================================================
# 8. Thinking 128 실행
# ============================================================

OUT_PATH = Path(
    "/content/drive/MyDrive/cleaning_data/"
    "ling3_vl_int4_eval/"
    "ling3_vl_int4_thinking_128.csv"
)


records = []

start_all = time.time()


print("\n" + "=" * 80)
print("LING THINKING — FIXED 128")
print("=" * 80)


for i, row in bench.iterrows():

    t0 = time.time()

    gt = (
        str(row["answer"])
        .strip()
        .lower()
    )


    row_run = row.copy()

    row_run["path"] = str(
        resolve_img(
            row["path"]
        )
    )


    pred = ""
    raw = ""
    status = "error"
    error = ""


    for max_side in [
        None,
        1600,
        1280,
        1024,
    ]:

        try:

            pred, raw = (
                infer_one_thinking(
                    row_run,
                    max_side=max_side,
                    return_raw=True,
                )
            )

            status = "ok"
            error = ""

            break

        except Exception as e:

            error = repr(e)

            gc.collect()

            torch.cuda.empty_cache()


    elapsed = (
        time.time()
        - t0
    )


    direct_pred = direct_map.get(
        str(row["id"]),
        ""
    )


    records.append({

        "idx":
            int(i),

        "id":
            str(row["id"]),

        "gt":
            gt,

        "direct":
            direct_pred,

        "thinking":
            pred,

        "direct_correct":
            direct_pred == gt,

        "thinking_correct":
            pred == gt,

        "status":
            status,

        "elapsed_sec":
            round(elapsed, 4),

        "raw":
            raw,

        "error":
            error,
    })


    if (
        (i + 1) % 8 == 0
        or i == len(bench) - 1
    ):

        tmp = pd.DataFrame(
            records
        )

        tmp.to_csv(
            OUT_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        ok_tmp = (
            tmp["status"]
            .eq("ok")
        )

        acc = (
            tmp.loc[
                ok_tmp,
                "thinking_correct"
            ].mean()
        )

        avg = (
            tmp.loc[
                ok_tmp,
                "elapsed_sec"
            ].mean()
        )

        print(
            f"[{i+1:03d}/128] "
            f"Thinking ACC={acc:.4f} | "
            f"{avg:.3f}s/개"
        )


# ============================================================
# 9. 최종 비교
# ============================================================

res = pd.DataFrame(
    records
)

ok = (
    res["status"]
    .eq("ok")
)

r = res[
    ok
].copy()


direct_acc = (
    r["direct_correct"]
    .mean()
)

thinking_acc = (
    r["thinking_correct"]
    .mean()
)


rescued = (
    (~r["direct_correct"])
    & r["thinking_correct"]
)

harmed = (
    r["direct_correct"]
    & (~r["thinking_correct"])
)

both_correct = (
    r["direct_correct"]
    & r["thinking_correct"]
)

both_wrong = (
    (~r["direct_correct"])
    & (~r["thinking_correct"])
)

disagree = (
    r["direct"]
    != r["thinking"]
)


print("\n" + "=" * 80)
print("DIRECT vs THINKING")
print("=" * 80)

print(
    f"Direct   : "
    f"{int(r['direct_correct'].sum())}/{len(r)} "
    f"= {direct_acc:.6f}"
)

print(
    f"Thinking : "
    f"{int(r['thinking_correct'].sum())}/{len(r)} "
    f"= {thinking_acc:.6f}"
)

print(
    f"Delta    : "
    f"{thinking_acc-direct_acc:+.6f}"
)

print()

print(
    "Rescued       :",
    int(rescued.sum())
)

print(
    "Harmed        :",
    int(harmed.sum())
)

print(
    "Net           :",
    int(rescued.sum() - harmed.sum())
)

print(
    "Both correct  :",
    int(both_correct.sum())
)

print(
    "Both wrong    :",
    int(both_wrong.sum())
)

print(
    "Disagreement  :",
    int(disagree.sum())
)

print(
    "Parse/errors  :",
    int((~ok).sum())
)

print(
    "Thinking avg s:",
    f"{r['elapsed_sec'].mean():.3f}"
)

print(
    "Total wall    :",
    f"{time.time()-start_all:.1f}s"
)

print(
    "Saved         :",
    OUT_PATH
)

print("=" * 80)


# ============================================================
# 10. Direct/Thinking이 바뀐 문제 확인
# ============================================================

changed = r.loc[
    disagree,
    [
        "id",
        "gt",
        "direct",
        "thinking",
        "direct_correct",
        "thinking_correct",
        "elapsed_sec",
    ]
].copy()


print(
    "\nChanged:",
    len(changed)
)

display(
    changed
)

✅ infer_one_thinking 준비 완료

LING THINKING — FIXED 128
[008/128] Thinking ACC=0.7500 | 11.857s/개
[016/128] Thinking ACC=0.6875 | 11.974s/개
[024/128] Thinking ACC=0.6667 | 11.986s/개
[032/128] Thinking ACC=0.6875 | 12.195s/개
[040/128] Thinking ACC=0.7000 | 11.717s/개
[048/128] Thinking ACC=0.7083 | 11.302s/개


KeyboardInterrupt: 

## Step 4. Test 6,714개 전체 추론 + 제출


In [6]:
# ============================================================
# STEP 4. FULL TEST 6,714 + RESUME + SUBMISSION
#
# - Ling-3.0-flash-VL-int4
# - Thinking OFF (기존 infer_one 그대로 사용)
# - zero-shot
# - 원본 이미지 우선
# - 실패 문항만 1600 -> 1280 -> 1024 자동 재시도
# - 25문항마다 Drive checkpoint 저장
# - 같은 셀 재실행 시 완료된 ID 자동 skip
# ============================================================

import gc
import os
import time
from pathlib import Path

import pandas as pd
import torch


# ============================================================
# 0. THINKING 테스트 후 메모리 정리
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "CUDA free:",
    f"{torch.cuda.mem_get_info()[0] / 2**30:.2f} GiB"
)


# ============================================================
# 1. OUTPUT PATH
# ============================================================

OUT_DIR = Path(
    "/content/drive/MyDrive/cleaning_data/"
    "ling3_vl_int4_eval"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_CSV = (
    OUT_DIR
    / "ling3_vl_int4_test_predictions.csv"
)

SUBMISSION_CSV = (
    OUT_DIR
    / "ling3_vl_int4_submission.csv"
)


# ============================================================
# 2. RESUME
# ============================================================

results = {}

if RESULT_CSV.exists():

    old = pd.read_csv(
        RESULT_CSV,
        dtype={"id": str},
    )

    for rec in (
        old
        .drop_duplicates(
            "id",
            keep="last",
        )
        .to_dict("records")
    ):

        results[
            str(rec["id"])
        ] = rec


done_ids = {

    k

    for k, v in results.items()

    if (
        str(
            v.get(
                "status",
                "",
            )
        ) == "ok"

        and

        str(
            v.get(
                "prediction",
                "",
            )
        )
        .strip()
        .lower()

        in {
            "a",
            "b",
            "c",
            "d",
        }
    )
}


# ============================================================
# 3. SAVE HELPER
# ============================================================

def save_results():

    if not results:
        return

    x = (
        pd.DataFrame(
            list(
                results.values()
            )
        )
        .sort_values("idx")
        .reset_index(drop=True)
    )

    tmp = Path(
        str(RESULT_CSV)
        + ".tmp"
    )

    x.to_csv(
        tmp,
        index=False,
        encoding="utf-8-sig",
    )

    os.replace(
        tmp,
        RESULT_CSV,
    )


def hms(sec):

    sec = int(sec)

    h, r = divmod(
        sec,
        3600,
    )

    m, s = divmod(
        r,
        60,
    )

    return (
        f"{h:02d}:"
        f"{m:02d}:"
        f"{s:02d}"
    )


# ============================================================
# 4. FULL TEST
# ============================================================

print("=" * 78)

print(
    "LING3_VL_INT4 ZERO-SHOT FULL TEST"
)

print(
    "RESUME:",
    len(done_ids),
    "/",
    len(test_df),
)

print(
    "RESULT:",
    RESULT_CSV,
)

print("=" * 78)


session_start = time.time()

session_done = 0

session_fail = 0


for idx, row in test_df.iterrows():

    img_id = str(
        row["id"]
    )


    # 이미 성공한 문항 skip
    if img_id in done_ids:
        continue


    pred = ""

    status = "error"

    error = ""

    used_max_side = "original"


    t0 = time.time()


    # ========================================================
    # OCR가 중요한 데이터라 원본 이미지 우선
    #
    # 오류/OOM/context 문제 발생 시에만
    # 순차적으로 축소
    # ========================================================

    retry_sizes = [
        None,
        1600,
        1280,
        1024,
    ]


    for max_side in retry_sizes:

        try:

            pred = infer_one(
                row,
                max_side=max_side,
            )


            status = "ok"

            error = ""


            used_max_side = (
                "original"
                if max_side is None
                else str(max_side)
            )


            break


        except Exception as e:

            error = repr(e)


            gc.collect()


            if torch.cuda.is_available():
                torch.cuda.empty_cache()


    elapsed = (
        time.time()
        - t0
    )


    # ========================================================
    # 결과 기록
    # ========================================================

    results[
        img_id
    ] = {

        "idx":
            int(idx),

        "id":
            img_id,

        "path":
            row["path"],

        "prediction":
            pred,

        "status":
            status,

        "elapsed_sec":
            round(
                elapsed,
                4,
            ),

        "used_max_side":
            used_max_side,

        "error":
            error,
    }


    if status == "ok":

        done_ids.add(
            img_id
        )

    else:

        session_fail += 1


    session_done += 1


    # ========================================================
    # 25문항마다 Drive checkpoint
    # ========================================================

    if (
        session_done % 25
        == 0
    ):

        save_results()


    # ========================================================
    # 진행상황 출력
    # ========================================================

    if (
        session_done == 1

        or

        session_done % 25 == 0

        or

        len(done_ids)
        == len(test_df)
    ):

        elapsed_all = (
            time.time()
            - session_start
        )


        avg = (
            elapsed_all
            / max(
                session_done,
                1,
            )
        )


        remain = (
            len(test_df)
            - len(done_ids)
        )


        print(

            f"[{len(done_ids):,}/"
            f"{len(test_df):,}] "

            f"{len(done_ids)/len(test_df):.1%} | "

            f"{avg:.2f}s/개 | "

            f"경과 {hms(elapsed_all)} | "

            f"ETA {hms(avg * remain)} | "

            f"fail={session_fail}",

            flush=True,
        )


# ============================================================
# 5. 마지막 checkpoint
# ============================================================

save_results()


# ============================================================
# 6. FINAL VALIDATION
# ============================================================

final = (
    pd.DataFrame(
        list(
            results.values()
        )
    )
    .sort_values("idx")
    .reset_index(drop=True)
)


valid = (

    final["status"]
    .eq("ok")

    &

    final[
        "prediction"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        [
            "a",
            "b",
            "c",
            "d",
        ]
    )
)


print(
    "\nTOTAL:",
    len(final),
)

print(
    "VALID:",
    int(
        valid.sum()
    ),
)

print(
    "ERROR:",
    int(
        (~valid).sum()
    ),
)


# ============================================================
# 7. 오류가 있으면 중단
# ============================================================

if (
    len(final)
    != len(test_df)

    or

    not valid.all()
):

    bad = final.loc[
        ~valid,
        [
            "idx",
            "id",
            "path",
            "error",
        ]
    ]


    display(
        bad.head(20)
    )


    raise RuntimeError(

        "미완료/오류 문항이 있습니다. "
        "같은 셀을 다시 실행하면 "
        "정상 완료된 ID는 건너뛰고 "
        "실패한 문제만 재시도합니다."
    )


# ============================================================
# 8. SUBMISSION 생성
# ============================================================

pred_map = dict(
    zip(

        final[
            "id"
        ].astype(str),

        final[
            "prediction"
        ]
        .astype(str)
        .str.strip()
        .str.lower(),
    )
)


submission = (
    sample_submission[
        ["id"]
    ]
    .copy()
)


submission[
    "answer"
] = (

    submission[
        "id"
    ]
    .astype(str)
    .map(
        pred_map
    )
)


# ============================================================
# 9. SUBMISSION VALIDATION
# ============================================================

assert (
    len(submission)
    == len(test_df)
)

assert (
    submission[
        "answer"
    ]
    .notna()
    .all()
)

assert (
    submission[
        "answer"
    ]
    .isin(
        [
            "a",
            "b",
            "c",
            "d",
        ]
    )
    .all()
)


# ============================================================
# 10. SAVE SUBMISSION
# ============================================================

submission.to_csv(
    SUBMISSION_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 11. SUMMARY
# ============================================================

total_wall = (
    time.time()
    - session_start
)


print(
    "\n"
    + "=" * 78
)

print(
    "SUBMISSION READY"
)

print(
    "PATH:",
    SUBMISSION_CSV,
)

print(
    "ROWS:",
    len(submission),
)

print(
    "TOTAL TIME:",
    hms(total_wall),
)

print(
    "AVG:",
    f"{total_wall / max(session_done, 1):.3f}s/개",
)

print(
    "FAIL:",
    session_fail,
)

print(
    "\nDISTRIBUTION:"
)

print(
    submission[
        "answer"
    ]
    .value_counts()
    .sort_index()
)

print(
    "=" * 78
)


display(
    submission.head(10)
)

CUDA free: 9.46 GiB
LING3_VL_INT4 ZERO-SHOT FULL TEST
RESUME: 0 / 6714
RESULT: /content/drive/MyDrive/cleaning_data/ling3_vl_int4_eval/ling3_vl_int4_test_predictions.csv
[1/6,714] 0.0% | 5.01s/개 | 경과 00:00:05 | ETA 09:20:19 | fail=0
[25/6,714] 0.4% | 0.37s/개 | 경과 00:00:09 | ETA 00:40:46 | fail=0
[50/6,714] 0.7% | 0.27s/개 | 경과 00:00:13 | ETA 00:29:50 | fail=0
[75/6,714] 1.1% | 0.24s/개 | 경과 00:00:17 | ETA 00:26:22 | fail=0
[100/6,714] 1.5% | 0.22s/개 | 경과 00:00:22 | ETA 00:24:30 | fail=0
[125/6,714] 1.9% | 0.21s/개 | 경과 00:00:26 | ETA 00:23:23 | fail=0
[150/6,714] 2.2% | 0.21s/개 | 경과 00:00:31 | ETA 00:22:39 | fail=0
[175/6,714] 2.6% | 0.20s/개 | 경과 00:00:35 | ETA 00:22:02 | fail=0
[200/6,714] 3.0% | 0.20s/개 | 경과 00:00:39 | ETA 00:21:41 | fail=0
[225/6,714] 3.4% | 0.20s/개 | 경과 00:00:44 | ETA 00:21:18 | fail=0
[250/6,714] 3.7% | 0.19s/개 | 경과 00:00:48 | ETA 00:20:59 | fail=0
[275/6,714] 4.1% | 0.19s/개 | 경과 00:00:53 | ETA 00:20:42 | fail=0
[300/6,714] 4.5% | 0.19s/개 | 경과 00:00:57 | ETA 00:20:29

,id,answer
0,test_0001.jpg,b
1,test_0002.jpg,c
2,test_0003.jpg,d
3,test_0004.jpg,d
4,test_0005.jpg,c
5,test_0006.jpg,d
6,test_0007.jpg,b
7,test_0008.jpg,c
8,test_0009.jpg,c
9,test_0010.jpg,a
